# D43 VectorDB Code

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

last modified date : 2026.09
제작 : 박광석 (모두의연구소)

# 벡터 데이터베이스 비교와 온프레미스 세팅

이 노트에서는 세 종류의 벡터DB를 동일한 데이터로 비교한 뒤 Qdrant를 독립 서버 프로세스로 기동해 HTTP로 접속하고 인덱스 파라미터 튜닝, 벡터 양자화 압축, 무중단 재인덱싱, 스냅샷 복구 훈련을 차례대로 구현합니다.

지금까지는 파이썬 프로세스 안에서 도는 임베디드 모드를 썼다면 이번 실습에서는 처음으로 **벡터DB를 별도 서버 프로세스로 분리해 운영**합니다.

## 실행 환경

**실습 목표**
- 동일 벡터 데이터셋에서 넘파이 완전 탐색과 Qdrant·Chroma·Milvus 3종 벡터 DB의 인덱싱 시간, 검색 지연 시간, 누락 비율(`1 - recall@10`)을 측정해 비교할 수 있다.
- Qdrant 독립 서버에서 HNSW 파라미터(`ef`, `m`, `ef_construct`), Payload 인덱스, 스칼라·이진 양자화를 조정하며 성능 변화를 정량 검증할 수 있다.
- 컬렉션 별칭(Alias) 기반 무중단 재인덱싱과 스냅샷 백업 복구 훈련을 수행하여 복구 소요 시간(MTTR)을 측정할 수 있다.

이 노트는 **코랩 Pro+의 GPU 런타임**을 기준으로 씁니다. GPU는 임베딩 계산에만 쓰이므로 CPU 런타임에서도 동작하지만 Step 1의 임베딩 계산이 몇 배 길어집니다.

**올리는 모델은 임베딩 모델 multilingual-e5-small(384차원) 하나뿐이고 언어 모델은 올리지 않습니다.** 오늘은 검색만 다룹니다. 검색 결과로 답을 만드는 일은 다음 노트에서 하므로 "답변은 반드시 한국어로" 같은 출력 언어 고정도 나오지 않습니다.

**API 키는 필요 없습니다.** 벡터DB 세 개가 전부 이 런타임 안에서 실행됩니다.

PC나 사내 서버에서 해야 하는 일에는 **[현업 환경 적용]** 표시를 붙였습니다. 코랩에서는 읽고 넘어가도 됩니다.

<div style="font-size: 18px; line-height: 1.6;">

| 주요 소요 시간 구간 | 수행 작업 | 예상 소요 시간 |
|---|---|---|
| **Step 0 환경 준비** | 3종 벡터 DB 클라이언트·서버 바이너리 및 임베딩 패키지 설치 | 약 2~4분 |
| **Step 1 코퍼스 임베딩** | 합성 청크 20,000건 임베딩 벡터 계산 (`multilingual-e5-small`) | GPU 2~4분 (CPU 15~30분) |
| **Step 2 Chroma·Milvus 비교** | 동일 벡터 20,000건을 각 DB에 적재하고 검색 지연·누락률 측정 | 약 2~5분 |
| **Step 3 Qdrant 서버 기동·적재** | Qdrant 서버 바이너리 기동 및 HTTP 배치 인덱싱 (20,000건) | 약 2~4분 |
| **Step 4.2 구축 파라미터 비교** | `m`·`ef_construct` 조합별 컬렉션 3회 재인덱싱 및 측정 | 약 3~6분 |
| **심화 트랙 1 규모별 교차점** | 5,000건 ~ 100,000건 규모별 완전 탐색 vs ANN 지연 측정 | 약 10~20분 |
| **심화 트랙 6 다중 벡터(Multi-Vector)** | 문서 8,000건을 문장 조각 24,000개로 분할 임베딩 및 2단계 검색 | 약 3~6분 |

</div>

시간이 모자라면 Step 1의 `N`을 5000으로 줄여도 됩니다. 필터 실습의 조건 만족 개수가 4분의 1이 되어 좁은 필터가 더 극단적으로 보일 뿐, 나머지 실습은 그대로 성립합니다.

디스크는 Qdrant 바이너리와 저장소, 다른 두 DB의 파일을 합쳐 **약 2GB**를 씁니다.

## Step 0. 환경 준비

In [1]:
import torch
from google.colab import drive
import os

drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/aiffel
os.chdir('/content/drive/MyDrive/aiffel')

Mounted at /content/drive


In [2]:
# 버전을 고정합니다. 상한이 없으면 어느 날 메이저 버전이 올라와 어제 돌던 코드가 멈춥니다
!pip install -q "qdrant-client>=1.19,<2" "chromadb>=1.5,<2" "pymilvus[milvus_lite]>=3.0,<4"
!pip install -q "langchain-huggingface>=1.2,<2" "sentence-transformers>=6,<7"

# pymilvus 3.x 부터는 임베디드 버전(milvus-lite)이 기본으로 함께 설치되지 않습니다.
# 대괄호 부분을 빼면 Step 2에서 "milvus-lite is required" 오류가 납니다

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 11.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 99.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.7/417.7 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 151.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.8/276.8 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 96.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━

In [3]:
import json, os, subprocess, time
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import requests


def pkgver(dist):
    """배포판 이름으로 설치된 버전을 읽습니다"""
    try:
        return version(dist)
    except PackageNotFoundError:
        return "미설치"


# qdrant_client 에는 __version__ 속성이 없습니다. 배포판 이름으로 읽어야 합니다
print("[설치된 버전]")
for d in ["qdrant-client", "chromadb", "pymilvus", "milvus-lite",
          "sentence-transformers", "langchain-huggingface"]:
    print(f"  {d:<24} {pkgver(d)}")

[설치된 버전]
  qdrant-client            1.19.1
  chromadb                 1.5.9
  pymilvus                 3.0.2
  milvus-lite              3.2.2
  sentence-transformers    6.2.0
  langchain-huggingface    1.2.3


## Step 1. 비교 실험용 합성 코퍼스 및 임베딩 생성

교재 : 4.5 벡터화

세 벡터DB를 공정하게 비교하려면 동일한 벡터를 넣어야 하므로 임베딩을 한 번 계산해 두고 재사용합니다. 또한 프라이빗 RAG 노트의 청크 12개 규모에서는 인덱스 간 차이가 드러나지 않으므로 이번에는 **2만 개 규모**로 합성 코퍼스를 생성합니다.

In [4]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

# 장치를 문자열로 고정해 두면 GPU 없는 환경에서 오류로 멈춥니다. 환경을 읽어 정합니다
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"임베딩 장치 : {DEVICE}")

emb = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",     # 384차원. 대량 실습에 적합
    # 바깥 model_kwargs 는 SentenceTransformer 로, 안쪽은 트랜스포머 모델로 갑니다.
    # torch_dtype= 은 폐기되었습니다. dtype= 을 씁니다
    model_kwargs={"device": DEVICE, "model_kwargs": {"dtype": "float32"}},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 128},
)

# 384를 상수로 고정하지 않고 모델에 한 번 물어봅니다.
# 모델을 바꾸면 이 줄이 따라 바뀌므로 아래 코드가 그대로 동작합니다
DIM = len(emb.embed_query("query: 차원 확인"))
print(f"임베딩 차원 {DIM}")

임베딩 장치 : cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

임베딩 차원 384


In [5]:
# 사내 문서를 모사한 청크를 대량 생성합니다
DEPTS = ["개발1팀", "개발2팀", "인프라팀", "기획팀", "영업팀", "보안팀", "인사팀"]
DOC_TYPES = ["규정", "매뉴얼", "회의록", "보고서", "지침"]
TOPICS = ["근무 시간", "재택근무", "연차 휴가", "경조 휴가", "정보 보안", "데이터 반출",
          "출장 여비", "장비 대여", "교육 지원", "성과 평가", "채용 절차", "예산 집행"]
VERBS = ["신청한다", "승인을 받아야 한다", "부서장에게 보고한다", "인사팀에 제출한다",
         "사전 협의가 필요하다", "매월 말일까지 제출한다"]

COMBO = len(DEPTS) * len(DOC_TYPES)      # 부서 7 × 종류 5 = 35개 조합
YEARS = 5


def make_chunks(n):
    """사내 문서를 모사한 청크를 만듭니다.

    주의할 것이 하나 있습니다. 부서와 종류가 i를 35로 나눈 나머지 하나로 정해지므로,
    연도를 i % 5 로 만들면 5가 35의 약수여서 **조합마다 연도가 하나로 고정**됩니다.
    그러면 "부서 + 종류 + 연도" 세 조건을 겹친 필터가 언제나 0건이 되어
    Step 5의 필터 실습이 통째로 성립하지 않습니다.
    그래서 연도는 35개 주기 **바깥**에서 바꿉니다.
    """
    rows = []
    for i in range(n):
        dept = DEPTS[i % len(DEPTS)]
        dtype = DOC_TYPES[(i // len(DEPTS)) % len(DOC_TYPES)]
        topic = TOPICS[(i // 3) % len(TOPICS)]
        verb = VERBS[i % len(VERBS)]
        year = 2022 + ((i // COMBO) % YEARS)
        text = (f"{dtype} 제{i%50+1}조 ({topic}) "
                f"{dept} 소속 임직원의 {topic} 관련 사항은 다음과 같다. "
                f"해당 절차는 {verb}. 본 조항은 {year}년 개정되었으며 "
                f"세부 기준은 별표 제{i%9+1}호에 따른다.")
        rows.append({
            "id": i,
            "text": text,
            "dept": dept,
            "doc_type": dtype,
            "topic": topic,
            "year": year,
        })
    return rows


N = 10000  # 실습 시간 단축을 위해 10,000건으로 경량화
chunks = make_chunks(N)
print(f"청크 {len(chunks)}개")
print(chunks[0]["text"])

청크 10000개
규정 제1조 (근무 시간) 개발1팀 소속 임직원의 근무 시간 관련 사항은 다음과 같다. 해당 절차는 신청한다. 본 조항은 2022년 개정되었으며 세부 기준은 별표 제1호에 따른다.


In [6]:
# 만든 데이터를 바로 검사합니다. 필드끼리 얽혀 있으면 필터 실습이 무의미해집니다
from collections import Counter


def combo_years(rows, dept, doc_type):
    """한 (부서, 종류) 조합 안에 연도가 몇 가지나 들어 있는지 셉니다"""
    return Counter(r["year"] for r in rows
                   if r["dept"] == dept and r["doc_type"] == doc_type)


dist = combo_years(chunks, "보안팀", "규정")
print("보안팀 규정의 연도 분포 :", dict(sorted(dist.items())))
print(f"연도 가짓수 {len(dist)} (1이면 세 조건 필터가 0건이 됩니다)")

보안팀 규정의 연도 분포 : {2022: 58, 2023: 57, 2024: 57, 2025: 57, 2026: 57}
연도 가짓수 5 (1이면 세 조건 필터가 0건이 됩니다)


데이터를 만들자마자 세어 봤습니다. 이론 5절에서 **결과가 이상하면 설정부터 바꾸지 말고 조건 만족 개수를 먼저 세라**고 한 습관은 데이터를 만드는 순간부터 적용됩니다. 합성 데이터에서 필드끼리 얽히면 아무 오류 없이 **필터 실습만 조용히 0건**이 됩니다.

In [7]:
# 임베딩 계산. 한 번만 하고 재사용합니다
t0 = time.time()
vectors = np.array(emb.embed_documents([f"passage: {c['text']}" for c in chunks]),
                   dtype=np.float32)
print(f"임베딩 완료 : {vectors.shape}, {time.time()-t0:.1f}초")
np.save("vectors.npy", vectors)

임베딩 완료 : (10000, 384), 10.0초


교재 4.5는 text-embedding-3-small을 예로 들어 청크에 비해 큰 모델은 비효율적일 수 있다고 합니다. 이 데이터의 청크 길이와 출력 차원별 메모리를 확인합니다(이론 1.5절).

In [8]:
# 교재 4.5 : 청크 길이와 출력 차원이 모델 선택과 메모리를 정합니다
lens = np.array([len(c["text"]) for c in chunks])
print(f"청크 길이 : 평균 {lens.mean():.0f}자, 최대 {lens.max()}자")
print(f"지금 벡터 : {vectors.nbytes / 1e6:.1f} MB = {N} × {DIM} × 4바이트\n")

mb_head = f"{N:,}개(MB)"
print(f"{'출력 차원':>8} {mb_head:>11} {'100만 개(GB)':>13}")
for d in sorted({DIM, 768, 1024, 1536, 3072}):
    note = "  <- 이 실습" if d == DIM else ("  <- 교재의 예" if d == 1536 else "")
    print(f"{d:>8} {N * d * 4 / 1e6:>11.1f} {1e6 * d * 4 / 1e9:>13.2f}{note}")


def estimate_capacity(n_docs=100_000, chunks_per_doc=10, dim=384, target_qps=20, ram_budget_gb=4.0):
    """문서 규모·차원·목표 QPS·서버 RAM 예산에 따른 온프레미스 벡터DB 용량 및 양자화 권고 산출기 (이론 1.2·3.5절)"""
    n_vecs = n_docs * chunks_per_doc
    bf_sec = (n_vecs * dim) / 1e10
    bf_max_qps = round(1.0 / max(bf_sec, 1e-6), 1)

    fp32_gb = (n_vecs * dim * 4) / 1e9
    sq8_gb = fp32_gb / 4.0
    bq_gb = fp32_gb / 32.0
    graph_gb = (n_vecs * 16 * 8) / 1e9  # M=16 기준 HNSW 그래프 간선 메모리 어림셈

    if fp32_gb + graph_gb <= ram_budget_gb * 0.7:
        rec = "FP32 원본 상주 (메모리 여유 충분, 최고 재현율)"
    elif sq8_gb + graph_gb <= ram_budget_gb * 0.7:
        rec = "스칼라 양자화 SQ8(INT8) + on_disk_payload=True (표준 운영 권장)"
    else:
        rec = "이진 양자화 BQ(1-bit) + 원본 디스크 보관(on_disk=True) + Oversampling 4배수 재점수화"

    print(f"\n[온프레미스 용량 산정 리포트] 총 벡터 {n_vecs:,}개 ({dim}차원), 예산 RAM {ram_budget_gb}GB")
    print(f"  - 1회 완전 탐색 시간 : 약 {bf_sec*1000:.1f}ms (단일 코어 최대 {bf_max_qps} QPS -> 목표 {target_qps} QPS 대비 {'ANN 필수' if bf_max_qps < target_qps * 2 else '완전 탐색 가능'})")
    print(f"  - FP32 원본 + 그래프 : {fp32_gb + graph_gb:.2f} GB (벡터 {fp32_gb:.2f}GB + 그래프 {graph_gb:.2f}GB)")
    print(f"  - SQ8(1/4) + 그래프  : {sq8_gb + graph_gb:.2f} GB")
    print(f"  - BQ(1/32) + 그래프  : {bq_gb + graph_gb:.2f} GB (원본 디스크 재점수화용 {fp32_gb:.2f}GB 별도)")
    print(f"  => 권장 구성 : {rec}")


# 현재 실습 규모(2만 개)와 현업 중형 규모(문서 10만 건 = 청크 100만 개, 1024차원) 비교 산정
estimate_capacity(n_docs=2_000, chunks_per_doc=10, dim=DIM, target_qps=10, ram_budget_gb=4.0)
estimate_capacity(n_docs=100_000, chunks_per_doc=10, dim=1024, target_qps=30, ram_budget_gb=4.0)

청크 길이 : 평균 106자, 최대 112자
지금 벡터 : 15.4 MB = 10000 × 384 × 4바이트

   출력 차원 10,000개(MB)    100만 개(GB)
     384        15.4          1.54  <- 이 실습
     768        30.7          3.07
    1024        41.0          4.10
    1536        61.4          6.14  <- 교재의 예
    3072       122.9         12.29

[온프레미스 용량 산정 리포트] 총 벡터 20,000개 (384차원), 예산 RAM 4.0GB
  - 1회 완전 탐색 시간 : 약 0.8ms (단일 코어 최대 1302.1 QPS -> 목표 10 QPS 대비 완전 탐색 가능)
  - FP32 원본 + 그래프 : 0.03 GB (벡터 0.03GB + 그래프 0.00GB)
  - SQ8(1/4) + 그래프  : 0.01 GB
  - BQ(1/32) + 그래프  : 0.00 GB (원본 디스크 재점수화용 0.03GB 별도)
  => 권장 구성 : FP32 원본 상주 (메모리 여유 충분, 최고 재현율)

[온프레미스 용량 산정 리포트] 총 벡터 1,000,000개 (1024차원), 예산 RAM 4.0GB
  - 1회 완전 탐색 시간 : 약 102.4ms (단일 코어 최대 9.8 QPS -> 목표 30 QPS 대비 ANN 필수)
  - FP32 원본 + 그래프 : 4.22 GB (벡터 4.10GB + 그래프 0.13GB)
  - SQ8(1/4) + 그래프  : 1.15 GB
  - BQ(1/32) + 그래프  : 0.26 GB (원본 디스크 재점수화용 4.10GB 별도)
  => 권장 구성 : 스칼라 양자화 SQ8(INT8) + on_disk_payload=True (표준 운영 권장)


청크가 100자 안팎이라 교재 4.5의 예처럼 최대 8,191토큰을 받아 8,000자 안팎의 청크에 맞는 1,536차원 모델은 이 데이터에 과하고 메모리만 네 배를 씁니다. 차원이 그대로 서버 메모리가 된다는 이론 1.2절의 어림셈은 모델을 고르는 단계부터 적용됩니다.

In [9]:
# 평가용 질의 세트와 정답(완전 탐색 결과)을 준비합니다.
# 이론 5.5절이 말한 "대표 질문 등록"이 이 목록입니다. 주제마다 말투를 바꿔 만듭니다
PATTERNS = ["{t} 관련 규정을 알려주세요",
            "{t}는 어떻게 신청하나요",
            "{t} 절차가 궁금합니다",
            "{t} 담당 부서와 제출 서류"]
queries = [p.format(t=t) for t in TOPICS for p in PATTERNS]

# 개수가 중요합니다. 이 목록이 이 노트의 **벤치마크 하네스**이고, 질문 수가 그 하네스의 해상도입니다.
# 질문이 8개면 문서 하나를 더 놓칠 때마다 측정값이 0.0125씩 뜁니다.
# 그러면 같은 데이터를 다시 인덱싱하기만 해도 값이 크게 흔들려 비교를 못 합니다
print(f"대표 질문 {len(queries)}개, 한 건을 놓칠 때 움직이는 폭 {1/(len(queries)*10):.4f}")

qvecs = np.array(emb.embed_documents([f"query: {q}" for q in queries]), dtype=np.float32)


def brute_force_topk(qv, k=10):
    """완전 탐색. ANN 검색이 무엇을 놓쳤는지 측정하는 기준이 됩니다"""
    scores = vectors @ qv
    return [int(i) for i in np.argsort(scores)[::-1][:k]]


ground_truth = {i: brute_force_topk(qvecs[i], k=10) for i in range(len(queries))}

t0 = time.time()
for i in range(len(queries)):
    brute_force_topk(qvecs[i], k=10)
bf_ms = (time.time() - t0) / len(queries) * 1000
print(f"완전 탐색 : {bf_ms:.1f} ms/질의 (기준선)")
print(f"질의 0의 정답 상위 5 : {ground_truth[0][:5]}")

대표 질문 48개, 한 건을 놓칠 때 움직이는 폭 0.0021
완전 탐색 : 0.4 ms/질의 (기준선)
질의 0의 정답 상위 5 : [5114, 2664, 8964, 144, 6444]


2만 개 규모에서는 넘파이 완전 탐색이 수 밀리초 안에 끝납니다. 원본 벡터 메모리도 `20,000 × 384 × 4바이트 ≈ 30.7MB`에 불과합니다.

이론 1.2절의 메모리 산정식($N \times d \times 4\text{B}$)으로 자릿수를 먼저 확인하면, 단일 프로세스 실험에서는 넘파이 배열만으로 충분하고 문서 규모가 수십만~수백만 개로 커지거나 다중 워커 동시 요청(QPS)·복합 메타데이터 필터링·무중단 갱신이 필요할 때부터 벡터 DB 인덱스가 위력을 발휘한다는 점을 명확히 알 수 있습니다.

## Step 2. 3종 벡터 DB(Qdrant·Chroma·Milvus) 성능 비교

교재 : 4.5.1 벡터 데이터베이스, 표 4-1 벡터 데이터베이스 선택

같은 벡터를 세 DB에 넣고 인덱싱 시간, 검색 지연, 놓치는 비율을 비교합니다. 놓치는 비율은 완전 탐색의 상위 10개 중 빠뜨린 비율, 곧 1 − 재현율(recall@10)입니다(이론 2.1절). 교재 표 4-1의 지표 가운데 검색 지연과 검색 정확도를 보는 것이고 처리량(QPS)은 동시 요청을 보내는 부하 시험이 따로 필요하고 수평 확장·복제는 서버가 여러 대 필요해 여기서는 다루지 않습니다. 필터 성능은 Step 5, 복구 시간은 Step 8-1에서 봅니다.

**놓치는 비율에는 바닥이 있습니다.** 이 합성 데이터는 같은 문장이 약 세 번씩 들어 있어(2만 개 중 문장 6,300종) 같은 벡터가 여럿이고 id로 비교하는 놓치는 비율은 점수가 같은 청크 중 다른 것을 고르면 놓친 것으로 셉니다. 그래서 완전 탐색을 하는 DB도 동점을 고르는 순서가 다르면 0.07 안팎으로 나타나고 이 값이 이 데이터에서 놓치는 비율의 바닥입니다. 값은 이 바닥과 견줘 읽고 동점을 인정해 다시 세는 확인은 Step 5 끝의 표에서 합니다.

In [10]:
def miss_rate(got, truth, k=10):
    """ANN 검색이 완전 탐색의 상위 k개 중 몇 퍼센트를 놓쳤는지. 1 - 재현율(recall@k)입니다.

    0이면 하나도 안 놓친 것이고, 0.3이면 열 개 중 셋을 빠뜨린 것입니다.
    이론 2.1절의 검사를 그대로 옮긴 것입니다.
    """
    return 1 - len(set(got[:k]) & set(truth[:k])) / k

In [11]:
# Chroma
import chromadb

t0 = time.time()
chroma = chromadb.PersistentClient(path="./chroma_db")
try:
    chroma.delete_collection("docs")
except Exception:
    pass                      # 없는 컬렉션을 지우려 한 것은 실패가 아닙니다
col = chroma.create_collection("docs", metadata={"hnsw:space": "cosine"})

B = 2000
for s in range(0, N, B):
    e = min(s + B, N)
    col.add(ids=[str(c["id"]) for c in chunks[s:e]],
            embeddings=vectors[s:e].tolist(),
            metadatas=[{k: c[k] for k in ("dept", "doc_type", "topic", "year")}
                       for c in chunks[s:e]])
chroma_index_sec = time.time() - t0
print(f"Chroma 인덱싱 : {chroma_index_sec:.1f}초")

Chroma 인덱싱 : 11.4초


In [13]:
t0 = time.time()
chroma_miss = []
for i in range(len(queries)):
    r = col.query(query_embeddings=[qvecs[i].tolist()], n_results=10)
    got = [int(x) for x in r["ids"][0]]
    chroma_miss.append(miss_rate(got, ground_truth[i]))
chroma_ms = (time.time() - t0) / len(queries) * 1000
print(f"Chroma 검색 : {chroma_ms:.1f} ms/질의, 놓치는 비율 {np.mean(chroma_miss):.3f}")

Chroma 검색 : 2.6 ms/질의, 놓치는 비율 0.008


In [14]:
# Milvus Lite
from pymilvus import MilvusClient

MILVUS_PATH = "./milvus_lite.db"
if os.path.exists(MILVUS_PATH):
    os.remove(MILVUS_PATH)

t0 = time.time()
mc = MilvusClient(MILVUS_PATH)
mc.create_collection(collection_name="docs", dimension=DIM, metric_type="COSINE",
                     auto_id=False)
for s in range(0, N, B):
    e = min(s + B, N)
    mc.insert(collection_name="docs",
              data=[{"id": c["id"], "vector": vectors[c["id"]].tolist(),
                     "dept": c["dept"], "doc_type": c["doc_type"],
                     "topic": c["topic"], "year": c["year"]}
                    for c in chunks[s:e]])
milvus_index_sec = time.time() - t0
print(f"Milvus Lite 인덱싱 : {milvus_index_sec:.1f}초")

ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
  File "/usr/local/lib/python3.13/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1357, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


Milvus Lite 인덱싱 : 2.4초


In [16]:
t0 = time.time()
milvus_miss = []
for i in range(len(queries)):
    r = mc.search(collection_name="docs", data=[qvecs[i].tolist()], limit=10,
                  output_fields=["id"])
    got = [x["id"] for x in r[0]]
    milvus_miss.append(miss_rate(got, ground_truth[i]))
milvus_ms = (time.time() - t0) / len(queries) * 1000
print(f"Milvus Lite 검색 : {milvus_ms:.1f} ms/질의, 놓치는 비율 {np.mean(milvus_miss):.3f}")

Milvus Lite 검색 : 1855.7 ms/질의, 놓치는 비율 0.012


## Step 3. Qdrant 독립 서버 프로세스 기동 및 인덱싱

지금까지 임베디드 모드로 쓴 Qdrant를 **별도 프로세스로 실행해 네트워크로 접속**합니다.

**[현업 환경 적용]** : 실무에서는 Docker로 실행합니다(`docker run -p 6333:6333 qdrant/qdrant`). 이 실습 환경에서는 컨테이너를 쓸 수 없어 **바이너리를 다운로드해 백그라운드 프로세스로 실행**합니다. 접속 방식과 API는 Docker로 실행한 것과 같으므로 이 노트의 코드에서 접속 주소만 사내 서버 주소로 바꾸면 그대로 동작합니다.

In [17]:
# Qdrant 실행 파일을 다운로드합니다
# 이론 4.4절에 적은 1.19 계열입니다. 1.16 이후의 메모리 계층과
# 필터 처리 개선이 들어 있으므로 이보다 낮은 버전으로 내리지 마세요
QDRANT_VER = "v1.19.1"
QDRANT_URL = "http://127.0.0.1:6333"
url = f"https://github.com/qdrant/qdrant/releases/download/{QDRANT_VER}/qdrant-x86_64-unknown-linux-gnu.tar.gz"

if not Path("qdrant").exists():
    !wget -q {url} -O qdrant.tar.gz
    !tar -xzf qdrant.tar.gz
    !chmod +x qdrant
!./qdrant --version

qdrant 1.19.1


In [18]:
# 설정 파일을 만듭니다
Path("qdrant_config").mkdir(exist_ok=True)
Path("qdrant_config/config.yaml").write_text("""
storage:
  storage_path: ./qdrant_storage
  snapshots_path: ./qdrant_snapshots
service:
  host: 127.0.0.1
  http_port: 6333
  grpc_port: 6334
telemetry_disabled: true
log_level: WARN
""".strip(), encoding="utf-8")

# 두 줄을 일부러 적었습니다.
#  snapshots_path : 기본값은 저장소 바깥의 ./snapshots 입니다. 백업 파일이 어디 있는지
#                   모르면 복구 훈련(Step 8-1)에서 파일을 못 찾습니다
#  telemetry_disabled : 기본값은 외부로 사용 통계를 보냅니다. 폐쇄망에서는 이 요청이
#                   실패하며 로그를 채우고, 보안 심사에서도 걸립니다
SNAP_DIR = Path("qdrant_snapshots")
print(Path("qdrant_config/config.yaml").read_text(encoding="utf-8"))

storage:
  storage_path: ./qdrant_storage
  snapshots_path: ./qdrant_snapshots
service:
  host: 127.0.0.1
  http_port: 6333
  grpc_port: 6334
telemetry_disabled: true
log_level: WARN


In [19]:
def start_qdrant(wait_sec=60):
    """서버를 기동하고 (프로세스, 상태 문자열)을 돌려줍니다.

    예외를 던지지 않습니다. 기동 실패는 오류가 아니라 **결과의 한 종류**로 다뤄야
    다음 셀에서 무엇을 확인할지 정할 수 있습니다.
    """
    log = open("qdrant.log", "a", encoding="utf-8")
    p = subprocess.Popen(["./qdrant", "--config-path", "qdrant_config/config.yaml"],
                         stdout=log, stderr=subprocess.STDOUT)
    for _ in range(wait_sec):
        if p.poll() is not None:
            return p, f"기동 실패 : 프로세스가 종료됨(코드 {p.returncode}). qdrant.log 를 확인하세요"
        try:
            if requests.get(f"{QDRANT_URL}/healthz", timeout=1).status_code == 200:
                return p, "기동 완료"
        except requests.exceptions.RequestException:
            pass
        time.sleep(1)          # 응답이 200이 아닐 때도 쉬어야 합니다
    return p, "기동 실패 : 시간 초과. qdrant.log 를 확인하세요"


proc, status = start_qdrant()
print(f"pid {proc.pid} : {status}")
if status != "기동 완료":
    !tail -20 qdrant.log

pid 7284 : 기동 완료


In [ ]:
# HTTP API로 직접 확인합니다. 이것이 임베디드 모드와의 결정적 차이입니다
r = requests.get(f"{QDRANT_URL}/")
print(json.dumps(r.json(), indent=2))

{
  "title": "qdrant - vector search engine",
  "version": "1.19.1",
  "commit": "6ab21cac18ebb6f4ae29102c7f8f5cc11affd5de"
}

**HTTP 서버가 생겨 달라지는 점**

- **여러 서비스가 공유**할 수 있습니다. 챗봇, 배치 파이프라인, 관리 도구가 같은 인덱스에 접근합니다  
- **애플리케이션과 독립적으로 재시작**됩니다. 챗봇을 배포해도 인덱스는 그대로입니다  
- **모니터링과 관리 API**를 쓸 수 있습니다. Step 8에서 확인합니다

In [20]:
# 클라이언트 접속. 프라이빗 RAG 노트와 달라진 것은 이 한 줄뿐입니다
from qdrant_client import QdrantClient
from qdrant_client.models import (Distance, VectorParams, PointStruct, HnswConfigDiff,
                                  Filter, FieldCondition, MatchValue, Range,
                                  SearchParams, ScalarQuantization, ScalarQuantizationConfig,
                                  ScalarType, PayloadSchemaType)

qc = QdrantClient(url=QDRANT_URL)
print(qc.get_collections())

collections=[]


In [21]:
def reset_collection(name, m=16, ef_construct=100, quantization=None, on_disk=False):
    """같은 이름으로 빈 컬렉션을 다시 만듭니다.

    recreate_collection 은 폐기 예정이라 삭제와 생성을 나눠서 합니다.
    """
    if qc.collection_exists(name):
        qc.delete_collection(name)
    qc.create_collection(
        collection_name=name,
        vectors_config=VectorParams(size=DIM, distance=Distance.COSINE, on_disk=on_disk),
        hnsw_config=HnswConfigDiff(m=m, ef_construct=ef_construct),
        quantization_config=quantization,
    )


def wait_green(name, timeout=600):
    """인덱싱이 끝나기를 기다리고 상태를 문자열로 돌려줍니다.

    무한 루프를 쓰지 않습니다. 인덱스가 red 로 떨어지면 영원히 green 이 되지 않는데,
    그때 셀이 멈춰 있으면 학생은 '느린 것'과 '망가진 것'을 구분할 수 없습니다.
    """
    t0 = time.time()
    while time.time() - t0 < timeout:
        st = qc.get_collection(name).status
        if st in ("green", "red"):
            return st
        time.sleep(1)
    return "시간 초과"


def create_and_index(name, m=16, ef_construct=100, quantization=None, on_disk=False,
                     items=None, vecs=None):
    """컬렉션을 만들고 벡터를 넣은 뒤 인덱싱이 끝날 때까지 기다립니다"""
    items = chunks if items is None else items
    vecs = vectors if vecs is None else vecs
    reset_collection(name, m=m, ef_construct=ef_construct,
                     quantization=quantization, on_disk=on_disk)
    t0 = time.time()
    for s in range(0, len(items), B):
        e = min(s + B, len(items))
        # 마지막 배치만 wait=True 로 보냅니다. 전부 wait=False 로 두면 서버가
        # 아직 적용하지 않은 배치가 남은 채로 상태가 green 이 되어,
        # **포인트 수가 모자란 컬렉션을 완성된 것으로 착각**하게 됩니다
        qc.upsert(collection_name=name, points=[
            PointStruct(id=c["id"], vector=vecs[c["id"]].tolist(),
                        payload={k: c[k] for k in ("text", "dept", "doc_type", "topic", "year")})
            for c in items[s:e]
        ], wait=(e >= len(items)))
    st = wait_green(name)
    if st != "green":
        print(f"  !! {name} 인덱스 상태 {st}. qdrant.log 를 확인하세요")
    return time.time() - t0


qdrant_index_sec = create_and_index("docs_base")
print(f"Qdrant 인덱싱 : {qdrant_index_sec:.1f}초")

Qdrant 인덱싱 : 4.4초


In [22]:
def qdrant_search(name, qv, k=10, ef=None, flt=None):
    params = SearchParams(hnsw_ef=ef) if ef else None
    hits = qc.query_points(collection_name=name, query=qv.tolist(), limit=k,
                           search_params=params, query_filter=flt).points
    return [h.id for h in hits]


t0 = time.time()
qdrant_miss = [miss_rate(qdrant_search("docs_base", qvecs[i]), ground_truth[i])
               for i in range(len(queries))]
qdrant_ms = (time.time() - t0) / len(queries) * 1000
print(f"Qdrant 검색 : {qdrant_ms:.1f} ms/질의, 놓치는 비율 {np.mean(qdrant_miss):.3f}")

Qdrant 검색 : 6.3 ms/질의, 놓치는 비율 0.012


In [23]:
print(f"{'DB':<16} {'인덱싱(초)':>10} {'검색(ms)':>10} {'놓치는 비율':>12}")
print("-" * 52)
print(f"{'완전 탐색':<16} {'-':>10} {bf_ms:>10.1f} {0.0:>12.3f}")
print(f"{'Chroma':<16} {chroma_index_sec:>10.1f} {chroma_ms:>10.1f} {np.mean(chroma_miss):>12.3f}")
print(f"{'Milvus Lite':<16} {milvus_index_sec:>10.1f} {milvus_ms:>10.1f} {np.mean(milvus_miss):>12.3f}")
print(f"{'Qdrant (서버)':<16} {qdrant_index_sec:>10.1f} {qdrant_ms:>10.1f} {np.mean(qdrant_miss):>12.3f}")

DB                   인덱싱(초)     검색(ms)       놓치는 비율
----------------------------------------------------
완전 탐색                     -        0.4        0.000
Chroma                 11.4        2.6        0.008
Milvus Lite             2.4     1855.7        0.012
Qdrant (서버)             4.4        6.3        0.012


**이 결과로 DB의 우열을 판단하면 안 됩니다.** 2만 개는 세 DB 모두에게 작은 규모입니다. Qdrant만 네트워크를 거치므로 지연에서 불리하고 Milvus Lite는 분산 기능이 빠진 경량 버전입니다.

**그래도 의미 있는 관찰**

- **놓치는 비율이 바닥값(0.07 안팎)에 붙은 DB는 이 규모에서 사실상 하나도 놓치지 않은 것입니다.** 제품에 따라서는 작은 데이터에서 인덱스를 쓰지 않고 완전 탐색하기도 합니다. 그편이 더 빠르고 정확하기 때문이고 이론 1절의 결론이 제품 안에도 들어 있습니다  
- **놓치는 비율이 바닥값보다 큰 DB에서 그 대가로 얻은 속도 이득은 이 규모에서 크지 않습니다.** 인덱스의 이득은 규모가 커질 때 나타납니다. 트랙 1에서 그 지점을 직접 찾습니다

## Step 4. HNSW 인덱스 파라미터(`ef`·`m`·`ef_construct`) 튜닝

교재 : 표 4-1 벡터 데이터베이스 선택

교재 표 4-1은 인덱싱 기법을 검색 정확도와 검색 속도로 평가합니다. 이론 2.3절의 설정 세 개를 직접 바꿔 정확도와 속도를 어떻게 맞바꾸는지 측정합니다. 제품마다 이름이 다르므로 대응부터 적어 둡니다.

<div style="font-size: 18px; line-height: 1.6;">

| 이론 2.3절의 HNSW 파라미터 | Qdrant API 설정 필드명 | 설정 적용 시점 |
|---|---|---|
| **`M` (노드당 최대 이웃 간선 수)** | `HnswConfigDiff(m=...)` | 컬렉션 생성 및 재인덱싱 시점 |
| **`ef_construct` (구축 시 탐색 후보 폭)** | `HnswConfigDiff(ef_construct=...)` | 컬렉션 생성 및 재인덱싱 시점 |
| **`ef` (질의 시 탐색 후보 폭)** | `SearchParams(hnsw_ef=...)` | 개별 검색 요청(`query_points`) 시점 |

</div>

#### 4.1 검색 시 후보 폭 ef (hnsw_ef)

가장 중요한 값이자 **재인덱싱 없이 조정할 수 있는 유일한 설정**입니다(이론 2.3절).

In [24]:
# 실습 : hnsw_ef 를 바꿔가며 놓치는 비율(miss_rate)과 지연(ms)을 측정하세요

def measure_ef(name, ef_values, k=10):
    rows = []
    for ef in ef_values:
        t0 = time.time()
        misses = []
        for i in range(len(queries)):
            ### START CODE HERE ### (검색 실행 및 놓치는 비율 계산 2줄)
            # 1) qdrant_search(name, qvecs[i], k=k, ef=ef) 로 검색 결과를 가져옵니다
            got = qdrant_search(name, qvecs[i], k=k, ef=ef)
            # 2) miss_rate(got, ground_truth[i], k) 로 정답 대비 놓친 비율을 계산합니다
            if got is not None:
                misses.append(miss_rate(got, ground_truth[i], k))
            ### END CODE HERE ###
        if not misses:
            print("[안내] measure_ef 내부의 got 빈칸을 채운 뒤 다시 실행하세요.")
            return []
        ms = (time.time() - t0) / len(queries) * 1000
        rows.append({"ef": ef, "miss": float(np.mean(misses)), "ms": ms})
    return rows

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
got = qdrant_search(name, qvecs[i], k=k, ef=ef)
misses.append(miss_rate(got, ground_truth[i], k))
```

**전형적인 곡선**

- **ef가 작을 때 많이 놓칩니다.** ef=8이면 후보를 너무 좁게 탐색해 정답을 빠뜨립니다  
- **ef를 올리면 놓치는 비율이 빠르게 떨어지다 바닥에 붙습니다.** 보통 64에서 128 사이에서 바닥값(이 데이터에서는 동점 때문에 0.07 안팎)에 닿고 그 이상 올려도 크게 나아지지 않습니다  
- **지연은 계속 늘어납니다.** 놓치는 비율이 바닥에 붙은 뒤에도 시간은 늘어납니다

그래서 최적점은 **놓치는 비율이 더 안 떨어지기 시작하는 지점**입니다. 그보다 높이면 비용만 늘어납니다. 검증 루프의 수렴 지점, 리랭커에 넘기는 후보 수, 시각 모델의 해상도에서도 반복해 만난 형태입니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

In [25]:
rows = measure_ef("docs_base", [8, 16, 32, 64, 128, 256, 512]) or []

print(f"{'ef':>8} {'놓치는 비율':>12} {'ms/질의':>10}")
print("-" * 34)
for row in rows:
    print(f"{row['ef']:>8} {row['miss']:>12.3f} {row['ms']:>10.2f}")
if not rows:
    print("(측정 결과가 비어 있습니다. 정답 코드를 붙였는지 확인하세요)")

      ef       놓치는 비율      ms/질의
----------------------------------
       8        0.012       6.05
      16        0.012       2.60
      32        0.012       2.53
      64        0.012       2.54
     128        0.012       2.47
     256        0.012       2.54
     512        0.012       2.67


#### 4.2 인덱스 구축 파라미터(`m`, `ef_construct`) 영향 측정

In [31]:
# 실습 : m 과 ef_construct 를 바꿔 인덱싱하고 인덱싱 시간과 최소 놓치는 비율을 비교하세요

configs = [
    {"name": "docs_m2", "m": 2, "ef_construct": 64},
    {"name": "docs_m4", "m": 4, "ef_construct": 64},
    {"name": "docs_m8", "m": 8, "ef_construct": 64},
    {"name": "docs_m16", "m": 16, "ef_construct": 100},
    {"name": "docs_m32", "m": 32, "ef_construct": 200},
]

results = []
for cfg in configs:
    ### START CODE HERE ### (컬렉션 인덱싱 및 동일 ef=128 조건의 놓치는 비율 계산 2줄)
    # 1) create_and_index(cfg["name"], m=cfg["m"], ef_construct=cfg["ef_construct"]) 를 호출합니다
    sec = create_and_index(cfg["name"], m=cfg["m"], ef_construct=cfg["ef_construct"])
    if sec is None:
        print("[안내] sec 와 misses 빈칸을 채운 뒤 실행하세요.")
        break
    # 2) 모든 질의에 대해 ef=128 로 검색한 결과의 miss_rate 리스트를 구합니다
    misses = [miss_rate(qdrant_search(cfg["name"], qvecs[i], ef=128), ground_truth[i])
              for i in range(len(queries))]
    ### END CODE HERE ###

    info = qc.get_collection(cfg["name"])
    results.append({
        "name": cfg["name"], "m": cfg["m"], "ef_c": cfg["ef_construct"],
        "index_sec": sec, "miss": float(np.mean(misses)),
        "points": info.points_count,
    })

if results:
    print(f"{'설정':<12} {'m':>4} {'ef_c':>6} {'인덱싱(초)':>10} {'놓치는 비율':>12}")
    print("-" * 50)
    for r in results:
        print(f"{r['name']:<12} {r['m']:>4} {r['ef_c']:>6} "
              f"{r['index_sec']:>10.1f} {r['miss']:>12.3f}")

설정              m   ef_c     인덱싱(초)       놓치는 비율
--------------------------------------------------
docs_m2         2     64        4.4        0.017
docs_m4         4     64        4.3        0.015
docs_m8         8     64        4.3        0.006
docs_m16       16    100        4.4        0.015
docs_m32       32    200        4.3        0.012


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
sec = create_and_index(cfg["name"], m=cfg["m"], ef_construct=cfg["ef_construct"])
misses = [miss_rate(qdrant_search(cfg["name"], qvecs[i], ef=128), ground_truth[i])
          for i in range(len(queries))]
```

**여기에도 정체 지점이 있습니다.**

m=8은 이웃이 너무 적어 그래프가 성기므로 눈에 띄게 많이 놓치고 m=16으로 올리면 크게 좋아집니다. 그런데 **m=32로 한 번 더 올리면 인덱싱 시간만 늘고 놓치는 비율은 거의 그대로**입니다. 4.1절의 곡선과 같은 형태입니다.

2만 개는 아직 작고 시간의 상당 부분이 벡터를 보내는 데 쓰여서 이 규모에서는 인덱싱 시간의 차이가 작게 보일 수 있습니다. **규모가 커지면 차이가 벌어지니** 그때 이 표를 다시 그려 봅니다.

**많이 놓칠 때는 먼저 ef를 올려 보고 그것으로 안 되면 m을 올립니다.** ef는 즉시 적용되지만 m은 재인덱싱이 필요합니다.

m을 올릴 때는 메모리 증가도 함께 계산합니다. 이론 3.1절에서 메모리가 나가는 두 군데를 봤습니다. 벡터 본체는 그대로이고 **이웃 목록이 m에 비례해 늘어납니다.**

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

## Step 5. 메타데이터 필터 통과율과 Payload 인덱스 성능 비교

교재 : 4.5.1 벡터 데이터베이스

이론 5.1절에서 사내 검색은 언제나 거르면서 찾는 일이라고 했습니다. 5.3절에서는 **먼저 거르면 느리고 나중에 거르면 놓치는 가운데 구간**을 봤고 교재 4.5.1은 메타데이터 조건으로 먼저 좁히면 질의 효율이 오른다고 합니다. 두 말이 각각 언제 맞는지 직접 확인합니다.

In [32]:
# 필터 없음 / 넓은 필터 / 중간 필터 / 좁은 필터를 비교합니다
filters = {
    "필터 없음": None,
    "넓은 필터 (부서)": Filter(must=[
        FieldCondition(key="dept", match=MatchValue(value="보안팀"))]),
    "중간 필터 (부서+종류)": Filter(must=[
        FieldCondition(key="dept", match=MatchValue(value="보안팀")),
        FieldCondition(key="doc_type", match=MatchValue(value="규정"))]),
    "좁은 필터 (부서+종류+연도)": Filter(must=[
        FieldCondition(key="dept", match=MatchValue(value="보안팀")),
        FieldCondition(key="doc_type", match=MatchValue(value="규정")),
        FieldCondition(key="year", range=Range(gte=2025))]),
}

print(f"{'조건':<28} {'결과수':>7} {'ms/질의':>10}")
print("-" * 48)
for label, flt in filters.items():
    t0 = time.time()
    counts = []
    for i in range(len(queries)):
        got = qdrant_search("docs_base", qvecs[i], k=10, ef=128, flt=flt)
        counts.append(len(got))
    ms = (time.time() - t0) / len(queries) * 1000
    print(f"{label:<28} {np.mean(counts):>7.1f} {ms:>10.2f}")

조건                               결과수      ms/질의
------------------------------------------------
필터 없음                           10.0       2.89
넓은 필터 (부서)                      10.0       8.72
중간 필터 (부서+종류)                   10.0       9.06
좁은 필터 (부서+종류+연도)                10.0       9.48


먼저 **지연**을 봅니다. 필터를 걸었을 뿐인데 몇 배 느려집니다. 이론 5.1절의 "조건을 붙이면 검색이 느려지거나 결과를 놓치는 경우" 가운데 앞의 경우가 그대로 재현되었습니다. 보통의 데이터베이스와 반대라 처음 보면 설정이 잘못된 줄 압니다.

**결과 수**는 데이터에 따라 줄기도 하고 안 줄기도 합니다. 10개를 다 채우지 못했다면 조건을 만족하는 문서가 애초에 적기 때문일 수도 있고 **인덱스 탐색이 조건에 맞는 후보까지 가지 못했기 때문**일 수도 있습니다.

이론 5절 Quiz에서 반드시 나눠야 한다고 한 두 가지입니다. 나누는 방법은 **조건 만족 문서 수를 직접 세는 것** 하나뿐입니다.

In [33]:
# 조건 만족 문서 수를 직접 셉니다
for label, flt in filters.items():
    if flt is None:
        print(f"{label:<28} 전체 {N}")
        continue
    cnt = qc.count(collection_name="docs_base", count_filter=flt, exact=True).count
    print(f"{label:<28} 조건 만족 {cnt}개 ({cnt/N:.2%})")

필터 없음                        전체 10000
넓은 필터 (부서)                   조건 만족 1428개 (14.28%)
중간 필터 (부서+종류)                조건 만족 286개 (2.86%)
좁은 필터 (부서+종류+연도)             조건 만족 114개 (1.14%)


세어본 결과가 **약 14% / 약 2.9% / 약 1.1%로** 나와야 합니다. 셋 다 0이 아닌지 먼저 확인합니다.

좁은 필터도 조건 만족 문서가 200개가 넘는데 필터 비교 셀에서 그 조건의 검색이 느렸습니다. **문서가 적어서가 아니라 거르면서 찾느라 느린 것**입니다. payload 인덱스가 없으면 제품이 조건 만족 개수를 미리 알 수 없고 조건을 확인하려면 점마다 payload를 읽어야 해서 느립니다(세 필터의 지연이 통과 비율과 상관없이 비슷합니다). 고칠 방법은 이론 5.4절에 있습니다.

In [34]:
# 실습 : payload 인덱스를 만들어 좁은 필터 성능을 개선하세요

narrow = filters["좁은 필터 (부서+종류+연도)"]


def measure_narrow(label):
    t0 = time.time()
    counts = []
    for i in range(len(queries)):
        got = qdrant_search("docs_base", qvecs[i], k=10, ef=128, flt=narrow)
        counts.append(len(got))
    ms = (time.time() - t0) / len(queries) * 1000
    print(f"{label:<16} 결과 {np.mean(counts):>5.1f}개  {ms:>7.2f} ms")


measure_narrow("인덱스 없음")

for field, schema in [("dept", PayloadSchemaType.KEYWORD),
                      ("doc_type", PayloadSchemaType.KEYWORD),
                      ("year", PayloadSchemaType.INTEGER)]:
    ### START CODE HERE ### ("docs_base" 컬렉션의 field에 schema 타입으로 payload 인덱스를 생성하세요)
    qc.create_payload_index(
        collection_name="docs_base",
        field_name=field,
        field_schema=schema,
        wait=True,
    )
    ### END CODE HERE ###

print("payload 인덱스 :", sorted(qc.get_collection("docs_base").payload_schema))
measure_narrow("인덱스 있음")

인덱스 없음           결과  10.0개     8.97 ms
payload 인덱스 : ['dept', 'doc_type', 'year']
인덱스 있음           결과  10.0개     2.49 ms


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
qc.create_payload_index(
    collection_name="docs_base",
    field_name=field,
    field_schema=schema,
    wait=True,
)
```

인덱스를 만들면 좁은 필터의 지연이 크게 줄어듭니다. 결과가 10개를 채우지 못했다면 결과 수도 늘어납니다. 이론 5.4절에서 살펴본 것처럼 필터 통과 문서 수가 매우 적을 때 Qdrant는 HNSW 그래프 탐색을 생략하고 **필터를 통과한 문서 벡터만 완전 탐색합니다.** 그렇게 판단하려면 조건에 맞는 것이 몇 개인지 알려 주는 payload 인덱스가 있어야 합니다. 인덱스가 없으면 제품은 판단하지 못하고 기본 경로로만 갑니다.

데이터 엔지니어링 노트와 파이프라인 노트에서 청크에 담은 부서, 등급, 버전으로 거는 필터가 전부 이 기능에 의존합니다.

**실무 규칙 : 필터에 쓰는 필드에는 반드시 payload 인덱스를 만듭니다.** 이론 5.5절의 운영 절차에도 들어 있습니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 5.1 네이티브 하이브리드 검색(Dense 의미 벡터 + Sparse 키워드 벡터 RRF 결합)

이론 5.5절에서 다룬 것처럼 사내 검색에서는 부서·연도 필터와 함께 **정확한 조항 번호·코드 일치(Sparse 벡터)**와 **문맥 의미 일치(Dense 벡터)**를 한 번의 쿼리로 결합합니다. Qdrant의 `SparseVectorParams`와 `Prefetch` + `FusionQuery(Fusion.RRF)`를 사용해 단일 요청으로 하이브리드 검색을 수행합니다.

In [35]:
# 네이티브 하이브리드 검색(Dense + Sparse RRF) 단일 쿼리 실행 (이론 5.5절)
from qdrant_client.models import SparseVectorParams, SparseVector, Prefetch, FusionQuery, Fusion
import zlib
import re


def text_to_sparse(text: str) -> SparseVector:
    """형태소·어절 토큰을 해시 인덱스와 빈도 가중치로 변환하는 경량 Sparse 벡터 생성기"""
    tokens = re.findall(r"[가-힣A-Za-z0-9]{2,}", text.lower())
    counts = Counter(zlib.crc32(tok.encode("utf-8")) % 50_000 for tok in tokens)
    indices = sorted(counts.keys())
    values = [float(counts[idx]) for idx in indices]
    return SparseVector(indices=indices, values=values)


HYB_COLL = "docs_hybrid"
if qc.collection_exists(HYB_COLL):
    qc.delete_collection(HYB_COLL)

qc.create_collection(
    collection_name=HYB_COLL,
    vectors_config={"dense": VectorParams(size=DIM, distance=Distance.COSINE)},
    sparse_vectors_config={"sparse": SparseVectorParams()},
)

HYB_SUB = 500
qc.upsert(
    collection_name=HYB_COLL,
    points=[
        PointStruct(
            id=c["id"],
            vector={
                "dense": vectors[c["id"]].tolist(),
                "sparse": text_to_sparse(c["text"]),
            },
            payload=c,
        )
        for c in chunks[:HYB_SUB]
    ],
    wait=True,
)

sample_q = queries[0]
q_dense = qvecs[0].tolist()
q_sparse = text_to_sparse(sample_q)

# 단일 네트워크 호출로 Dense 후보 20개 + Sparse 후보 20개를 추린 뒤 RRF(상호 순위 융합)로 상위 5개 확정
hyb_hits = qc.query_points(
    collection_name=HYB_COLL,
    prefetch=[
        Prefetch(query=q_dense, using="dense", limit=20),
        Prefetch(query=q_sparse, using="sparse", limit=20),
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=5,
).points

print(f"[하이브리드 RRF 검색] 질의: '{sample_q}'")
for rank, h in enumerate(hyb_hits, 1):
    print(f"  {rank}위 (RRF 점수 {h.score:.4f}) [{h.payload['dept']}/{h.payload['doc_type']}] {h.payload['text'][:55]}...")

[하이브리드 RRF 검색] 질의: '근무 시간 관련 규정을 알려주세요'
  1위 (RRF 점수 0.6250) [보안팀/규정] 규정 제11조 (근무 시간) 보안팀 소속 임직원의 근무 시간 관련 사항은 다음과 같다. 해당 절차는...
  2위 (RRF 점수 0.5000) [영업팀/규정] 규정 제45조 (근무 시간) 영업팀 소속 임직원의 근무 시간 관련 사항은 다음과 같다. 해당 절차는...
  3위 (RRF 점수 0.3333) [영업팀/규정] 규정 제25조 (근무 시간) 영업팀 소속 임직원의 근무 시간 관련 사항은 다음과 같다. 해당 절차는...
  4위 (RRF 점수 0.3333) [인프라팀/매뉴얼] 매뉴얼 제40조 (근무 시간) 인프라팀 소속 임직원의 근무 시간 관련 사항은 다음과 같다. 해당 절...
  5위 (RRF 점수 0.3214) [인사팀/규정] 규정 제17조 (근무 시간) 인사팀 소속 임직원의 근무 시간 관련 사항은 다음과 같다. 해당 절차는...


앞에서 본 교재 4.5.1의 말을 통과 비율별로 확인합니다. 정답은 넘파이로 같은 조건을 먼저 거른 뒤 완전 탐색한 결과이고 넘파이 열은 조건으로 거르는 일(조건마다 한 번)과 질문 전체의 완전 탐색을 함께 측정해 질문 하나로 나눈 시간입니다. "payload 인덱스 : 없음"으로 출력되면 위 정답 코드를 먼저 실행합니다.

통과 비율이 낮을수록 넘파이로 먼저 거르고 완전 탐색하는 시간은 남은 벡터 수에 비례해 줄고 결과는 언제나 정확합니다. 필터 인덱스가 있는 Qdrant도 통과 비율이 낮은 필터에서는 필터 없는 검색만큼 빠르거나 더 빠르고 동점을 인정하면 하나도 놓치지 않습니다. 교재 4.5.1의 말이 맞는 조건이 이것입니다. 조건이 좁고(통과 비율이 낮고) 제품이 조건 만족 개수를 알 때입니다. 인덱스 없이 측정한 앞의 비교 셀에서는 같은 필터가 몇 배 느렸습니다.

두 놓치는 비율이 다른 것은 Step 2에서 말한 동점 때문입니다. 필터 없음 줄의 놓치는 비율이 그 바닥값이고 동점을 인정하면 0에 가깝습니다.

절반 넘게 통과하는 조건에서 넘파이는 남은 60%를 여전히 전부 계산하므로 필터 없음일 때의 절반 넘게 걸리고 규모가 커지면 이론 5.2절에서 말한 대로 인덱스를 만든 의미가 사라집니다. Qdrant는 이 조건에서 그래프를 탐색하므로 지연 이득이 작고 재현율도 필터가 없을 때와 비슷하거나 조금 낮습니다. 이론 5.3절의 가운데 구간(먼저 거르면 느리고 나중에 거르면 놓치는 구간)은 조건 만족 벡터가 많아 제품이 그것만 완전 탐색하는 경로를 쓸 수 없는 규모에서 두드러지므로 2만 개 실습에서는 뚜렷하지 않을 수 있습니다.

## Step 6. 스칼라 양자화(SQ8) 적용과 메모리 절감 측정

교재 : 4.5 벡터화

이론 3.2절의 스칼라 양자화를 적용합니다. 교재 4.5가 말한 출력 차원의 비용과 달리 압축은 차원을 그대로 두고 값 하나의 크기를 줄입니다.

In [36]:
# 실습 : 스칼라 양자화(INT8)를 적용한 컬렉션을 만들고 원본과 비교하세요

def collection_size_mb(name):
    """컬렉션이 디스크에서 실제로 쓰는 크기 (희소 파일 왜곡 방지를 위해 st_blocks 사용)"""
    p = Path("qdrant_storage/collections") / name
    if not p.exists():
        return 0.0
    return sum(f.stat().st_blocks * 512 for f in p.rglob("*") if f.is_file()) / 1e6


### START CODE HERE ### (INT8 스칼라 양자화 설정을 완성하세요)
quant = ScalarQuantization(scalar=ScalarQuantizationConfig(
    type=ScalarType.INT8,
    quantile=0.99,      # 양쪽 끝의 극단값을 잘라 나눌 범위를 안정화합니다
    always_ram=True,    # 압축한 벡터를 메모리에 상주시킵니다
))
### END CODE HERE ###

if quant.scalar.type is None:
    print("[안내] ScalarQuantizationConfig의 type=ScalarType.INT8 빈칸을 채운 뒤 실행하세요.")
else:
    sec_q = create_and_index("docs_sq", m=16, ef_construct=100, quantization=quant)
    print(f"양자화 컬렉션 인덱싱 : {sec_q:.1f}초")

    base_mb = N * DIM * 4 / 1e6
    print(f"\n어림셈 : {N} × {DIM} × 4바이트 = {base_mb:.1f} MB")
    print(f"         값 하나를 1바이트로 줄이면 {base_mb/4:.1f} MB")

    print(f"\n{'컬렉션':<14} {'디스크(MB)':>11} {'놓치는 비율':>12} {'ms/질의':>10}")
    print("-" * 52)
    for name in ["docs_base", "docs_sq"]:
        t0 = time.time()
        misses = [miss_rate(qdrant_search(name, qvecs[i], ef=128), ground_truth[i])
                  for i in range(len(queries))]
        ms = (time.time() - t0) / len(queries) * 1000
        print(f"{name:<14} {collection_size_mb(name):>11.1f} "
              f"{np.mean(misses):>12.3f} {ms:>10.2f}")

양자화 컬렉션 인덱싱 : 4.3초

어림셈 : 10000 × 384 × 4바이트 = 15.4 MB
         값 하나를 1바이트로 줄이면 3.8 MB

컬렉션                디스크(MB)       놓치는 비율      ms/질의
----------------------------------------------------
docs_base            437.1        0.012       2.71
docs_sq              344.1        0.015       2.72


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
quant = ScalarQuantization(scalar=ScalarQuantizationConfig(
    type=ScalarType.INT8,
    quantile=0.99,
    always_ram=True,
))
```

**디스크 사용량이 오히려 늘어난 것**은 오류가 아닙니다. 이론 3.4절에서 살펴본 것처럼 2차 재점수화(Rescoring)를 수행하려면 원본 벡터가 필요하므로 양자화 컬렉션은 압축 벡터와 **원본 벡터를 디스크에 함께 보관**하기 때문입니다.

줄어드는 것은 디스크가 아니라 **검색이 상시 메모리에 올려두는 양**입니다. 그 값이 앞서 두 줄 어림셈으로 계산한 메모리 크기입니다. `always_ram=True`로 메모리에 올린 것은 4분의 1로 줄어든 쪽이고 원본은 디스크에 남아 있다가 재점수화할 때만 읽힙니다.

그래서 **압축의 이득을 디스크 사용량으로 보고하면 안 됩니다.** 보고할 값은 어림셈으로 계산한 벡터 본체 크기와 Step 8에서 볼 `memory_resident_bytes`입니다.

**더 놓치는 양은 우리 데이터로 측정해 봐야 압니다.** 이론 3.6절에서 말한 대로 벤더가 공개하는 "손실은 1퍼센트포인트 수준" 같은 수치는 공개 데이터셋에서 측정한 것입니다. 이 실습에서 눈에 띄게 벌어졌다면 여러분 데이터에서 재점수화가 필요하다는 신호입니다.

오버샘플링과 재점수화는 압축한 벡터로 후보를 넓게 뽑고 원본 벡터로 상위를 다시 정렬하는 방식입니다. `QuantizationSearchParams(oversampling=2.0, rescore=True)`처럼 두 값을 함께 지정합니다. 트랙 2에서 직접 만듭니다.

메모리에서 읽어 오는 양이 줄어 캐시 효율이 오르므로 **지연은 오히려 줄 수 있습니다.**

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

## Step 7. 컬렉션 별칭(Alias) 기반 무중단 재인덱싱

교재 : 4.5.2 최신 데이터 유지

이론 6.5절의 별칭은 운영 중인 서비스에서 임베딩 모델이나 인덱스 설정을 바꿀 때 쓰는 표준 방법입니다. 교재 4.5.2는 바뀐 데이터의 벡터를 다시 만들어 벡터DB API로 인덱스를 갱신하는 별도 파이프라인을 예로 듭니다. 바뀐 청크만 지우고 넣는 증분 반영은 파이프라인 노트 트랙 3(선택)에서 다루고 여기서는 모델이나 인덱스 설정이 바뀌어 전체를 다시 만들 때 쓰는 별칭 전환을 만듭니다.

In [37]:
# 실습 : 별칭(Alias)을 만들고, 새 컬렉션(docs_v2)으로 무중단 원자적 전환을 수행하세요

from qdrant_client.models import (CreateAliasOperation, CreateAlias,
                                  DeleteAliasOperation, DeleteAlias)

ALIAS = "docs_live"

# 1) 초기 별칭 생성 (docs_live -> docs_base)
qc.update_collection_aliases(change_aliases_operations=[
    CreateAliasOperation(create_alias=CreateAlias(
        collection_name="docs_base", alias_name=ALIAS))
])
print(f"별칭 {ALIAS} -> docs_base")


def service_search(q, k=3):
    """서비스 코드는 항상 별칭만 참조합니다"""
    qv = np.array(emb.embed_query(f"query: {q}"), dtype=np.float32)
    hits = qc.query_points(collection_name=ALIAS, query=qv.tolist(), limit=k,
                           search_params=SearchParams(hnsw_ef=128)).points
    return [(h.id, round(h.score, 3)) for h in hits]


print("전환 전 :", service_search("재택근무 신청 방법"))

# 3) 새 설정으로 컬렉션 인덱싱 (서비스는 계속 동작)
sec = create_and_index("docs_v2", m=32, ef_construct=200)
print(f"새 컬렉션 인덱싱 완료({sec:.1f}초). 이 동안 서비스는 중단되지 않았습니다")
print("인덱싱 중 검색 :", service_search("재택근무 신청 방법"))

### START CODE HERE ### (4단계: 별칭 삭제와 새 컬렉션("docs_v2") 별칭 생성을 한 리스트에 담아 원자적으로 전환)
target_new_collection = "docs_v2"
if target_new_collection is not None:
    qc.update_collection_aliases(change_aliases_operations=[
        DeleteAliasOperation(delete_alias=DeleteAlias(alias_name=ALIAS)),
        CreateAliasOperation(create_alias=CreateAlias(
            collection_name=target_new_collection, alias_name=ALIAS)),
    ])
### END CODE HERE ###

if target_new_collection is None:
    print("\n[안내] target_new_collection 빈칸에 'docs_v2'를 입력한 뒤 다시 실행하세요.")
else:
    print(f"\n별칭 전환 완료 : {ALIAS} -> {target_new_collection}")
    print("전환 후 :", service_search("재택근무 신청 방법"))
    print("\n현재 별칭 :", qc.get_aliases())

별칭 docs_live -> docs_base
전환 전 : [(41, 0.866), (6341, 0.866), (3891, 0.866)]
새 컬렉션 인덱싱 완료(4.5초). 이 동안 서비스는 중단되지 않았습니다
인덱싱 중 검색 : [(41, 0.866), (6341, 0.866), (3891, 0.866)]

별칭 전환 완료 : docs_live -> docs_v2
전환 후 : [(6341, 0.866), (41, 0.866), (3891, 0.866)]

현재 별칭 : aliases=[AliasDescription(alias_name='docs_live', collection_name='docs_v2')]


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
target_new_collection = "docs_v2"
qc.update_collection_aliases(change_aliases_operations=[
    DeleteAliasOperation(delete_alias=DeleteAlias(alias_name=ALIAS)),
    CreateAliasOperation(create_alias=CreateAlias(
        collection_name=target_new_collection, alias_name=ALIAS)),
])
```

**삭제와 생성을 한 번의 요청으로 보냅니다.** Qdrant가 원자적으로 처리하므로 별칭이 없는 순간이 존재하지 않습니다. 두 번의 요청으로 나누면 그 사이에 들어온 검색이 실패합니다(이론 6.5절).

**실무에서의 이점**

- **검증 후 전환** : 새 컬렉션을 인덱싱한 뒤 별칭을 옮기기 전에 직접 검색해 품질을 확인할 수 있습니다  
- **즉시 롤백** : 문제가 발견되면 재인덱싱 없이 별칭을 이전 컬렉션으로 되돌립니다  
- **서비스 코드 불변** : 애플리케이션은 별칭 이름만 알면 됩니다

지난 노트의 파이프라인과 결합하면 파이프라인이 청크를 생성해 새 컬렉션에 인덱싱하고 검증 게이트를 통과했을 때만 별칭을 전환합니다. 게이트가 실패하면 전환하지 않고 기존 인덱스를 유지합니다. 트랙 4에서 만듭니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

## Step 8. 서버 상태·메트릭 수집 및 스냅샷 점검

교재 : 표 4-1 벡터 데이터베이스 선택

서버 모드에서만 되는 관리 작업입니다. 교재 표 4-1의 장애 허용성(가동률, 복구 시간)은 상태와 메트릭을 감시하고 스냅샷을 남기는 데서 시작합니다.

In [38]:
# 컬렉션 정보와 상태
for name in ["docs_base", "docs_v2", "docs_sq"]:
    if not qc.collection_exists(name):
        print(f"{name:<12} 없음 (해당 실습을 건너뛰었다면 정상입니다)")
        continue
    info = qc.get_collection(name)
    print(f"{name:<12} 상태 {info.status}  포인트 {info.points_count}  "
          f"세그먼트 {info.segments_count}  필터 인덱스 {sorted(info.payload_schema)}")

docs_base    상태 green  포인트 10000  세그먼트 4  필터 인덱스 ['dept', 'doc_type', 'year']
docs_v2      상태 green  포인트 10000  세그먼트 4  필터 인덱스 []
docs_sq      상태 green  포인트 10000  세그먼트 4  필터 인덱스 []


In [44]:
from qdrant_client import QdrantClient

# 1. 타임아웃을 60초로 넉넉하게 늘린 클라이언트로 재연결
qc = QdrantClient("http://localhost:6333", timeout=60)

# 2. 원래 교재 코드 다시 실행
TARGET = "docs_v2" if qc.collection_exists("docs_v2") else "docs_base"
snap = qc.create_snapshot(collection_name=TARGET)
print(f"스냅샷 생성 성공! : {snap.name} ({snap.size/1e6:.1f} MB)")

# 파일 위치 확인
for f in sorted((SNAP_DIR / TARGET).glob("*.snapshot")):
  print(" ", f)

스냅샷 생성 성공! : docs_v2-2307530473306109-2026-10-22-07-31-09.snapshot (344.3 MB)
  qdrant_snapshots/docs_v2/docs_v2-2307530473306109-2026-10-22-07-22-42.snapshot
  qdrant_snapshots/docs_v2/docs_v2-2307530473306109-2026-10-22-07-23-03.snapshot
  qdrant_snapshots/docs_v2/docs_v2-2307530473306109-2026-10-22-07-26-07.snapshot
  qdrant_snapshots/docs_v2/docs_v2-2307530473306109-2026-10-22-07-29-24.snapshot
  qdrant_snapshots/docs_v2/docs_v2-2307530473306109-2026-10-22-07-31-09.snapshot


In [46]:
!ls -lh qdrant_snapshots/docs_v2

total 1.7G
-rw------- 1 root root 329M Oct 22 07:22 docs_v2-2307530473306109-2026-10-22-07-22-42.snapshot
-rw------- 1 root root   64 Oct 22 07:22 docs_v2-2307530473306109-2026-10-22-07-22-42.snapshot.checksum
-rw------- 1 root root 329M Oct 22 07:23 docs_v2-2307530473306109-2026-10-22-07-23-03.snapshot
-rw------- 1 root root   64 Oct 22 07:23 docs_v2-2307530473306109-2026-10-22-07-23-03.snapshot.checksum
-rw------- 1 root root 329M Oct 22 07:26 docs_v2-2307530473306109-2026-10-22-07-26-07.snapshot
-rw------- 1 root root   64 Oct 22 07:26 docs_v2-2307530473306109-2026-10-22-07-26-07.snapshot.checksum
-rw------- 1 root root 329M Oct 22 07:29 docs_v2-2307530473306109-2026-10-22-07-29-24.snapshot
-rw------- 1 root root   64 Oct 22 07:29 docs_v2-2307530473306109-2026-10-22-07-29-24.snapshot.checksum
-rw------- 1 root root 329M Oct 22 07:31 docs_v2-2307530473306109-2026-10-22-07-31-09.snapshot
-rw------- 1 root root   64 Oct 22 07:31 docs_v2-2307530473306109-2026-10-22-07-31-09.snapshot.che

In [45]:
# 메트릭 확인. 프로메테우스 같은 모니터링 시스템이 이 엔드포인트를 주기적으로 수집(Scraping)합니다
# 전부 찍으면 수백 줄이라 볼 것만 골라냅니다
WATCH = ("app_info", "collections_total", "collections_vector_total",
         "collection_points", "memory_resident_bytes", "snapshot_created_total")

r = requests.get(f"{QDRANT_URL}/metrics")
for line in r.text.split("\n"):
    if line and not line.startswith("#") and line.startswith(WATCH):
        print(line)

app_info{name="qdrant",version="1.19.1"} 1
collections_total 9
collections_vector_total 81000
collection_points{id="docs_base"} 10000
collection_points{id="docs_hybrid"} 500
collection_points{id="docs_m16"} 10000
collection_points{id="docs_m2"} 10000
collection_points{id="docs_m32"} 10000
collection_points{id="docs_m4"} 10000
collection_points{id="docs_m8"} 10000
collection_points{id="docs_sq"} 10000
collection_points{id="docs_v2"} 10000
snapshot_created_total{id="docs_v2"} 5
snapshot_created_total{id="docs_sq"} 0
snapshot_created_total{id="docs_m16"} 0
snapshot_created_total{id="docs_m2"} 0
snapshot_created_total{id="docs_m8"} 0
snapshot_created_total{id="docs_m32"} 0
snapshot_created_total{id="docs_hybrid"} 0
snapshot_created_total{id="docs_m4"} 0
snapshot_created_total{id="docs_base"} 0
memory_resident_bytes 93171712


**임계값을 걸어 둘 만한 것**

- **`memory_resident_bytes`** : 서버가 실제로 쓰는 메모리입니다. 이론 3절 Quiz에서 "메모리 사용률 70%를 넘으면 스칼라 양자화를 켠다"처럼 미리 정해 두라고 한 지표가 이것입니다  
- **`collection_points`** : 컬렉션별 포인트 수입니다. 재인덱싱 뒤에 이 값이 갑자기 줄었다면 인덱싱이 중간에 멈춘 것입니다

**[현업 환경 적용]** : 사내 운영 환경에서는 모니터링 서버(Prometheus 등)가 이 엔드포인트를 주기적으로 수집(Scraping)하도록 설정합니다. 프로덕션 주간에서 그 등록을 직접 합니다.

임베디드 모드에는 이런 것이 없습니다. **운영 가시성이 서버 모드의 핵심 이점**입니다.

### Step 8-1. 스냅샷 기반 컬렉션 복구 및 MTTR 측정
교재 : 표 4-1 벡터 데이터베이스 선택

교재 표 4-1은 장애 허용성을 가동률과 복구 시간(MTTR)으로 측정합니다. 서버가 한 대인 실습에서는 데이터 복제나 고가용성을 만들 수 없으므로(복제본에서 생기는 문제는 이론 6.2절) 스냅샷에서 컬렉션을 되살리는 데 걸리는 시간을 측정합니다.

교재 4.4.9가 말하는 정기 백업도 실제로 복구해 봐야 의미가 있습니다. 이론 마치며의 당부대로 해보지 않은 백업은 백업이 아닙니다.

In [47]:
def ensure_qdrant():
    global proc, qc
    try:
        qc.get_collections()
    except Exception:
        proc, _st = start_qdrant()
        qc = QdrantClient(url=QDRANT_URL)
        print(f"Qdrant 서버 자동 재기동 완료: {_st}")

ensure_qdrant()

# 실습 : 스냅샷 백업 생성 및 컬렉션 유실 후 복구·별칭 재연결 훈련을 수행하세요

from qdrant_client.models import CreateAliasOperation, CreateAlias

RECOVER = next((a.collection_name for a in qc.get_aliases().aliases
                if a.alias_name == ALIAS), "docs_base")
print(f"복구 대상 : {RECOVER}")

# 0) 복구 전 상태 기록
before = qdrant_search(RECOVER, qvecs[0], k=5, ef=128)
n_before = qc.get_collection(RECOVER).points_count
alias_before = [a.alias_name for a in qc.get_aliases().aliases
                if a.collection_name == RECOVER]
print(f"복구 전 : {n_before}개, 질의 0 결과 {before}, 별칭 {alias_before}")

# 1) 스냅샷 생성
t0 = time.time()
### START CODE HERE ### (RECOVER 컬렉션의 스냅샷을 생성하세요)
snap = qc.create_snapshot(collection_name=RECOVER)
### END CODE HERE ###

if snap is None:
    print("[안내] snap 빈칸을 채운 뒤 다시 실행하세요.")
else:
    t_snap = time.time() - t0
    snap_path = SNAP_DIR / RECOVER / snap.name
    print(f"스냅샷 : {snap_path} ({snap.size/1e6:.1f} MB), 파일 있음 {snap_path.exists()}")

    # 2) 장애 재현 : 컬렉션 삭제
    qc.delete_collection(RECOVER)
    t_down = time.time()
    print(f"컬렉션 삭제됨. 존재 여부 {qc.collection_exists(RECOVER)}, "
          f"별칭 {qc.get_aliases().aliases}")

    # 3) 스냅샷 업로드로 복구
    t0 = time.time()
    with open(snap_path, "rb") as f:
        r = requests.post(
            f"{QDRANT_URL}/collections/{RECOVER}/snapshots/upload?priority=snapshot",
            files={"snapshot": (snap.name, f)}, timeout=600)
    t_upload = time.time() - t0
    print(f"복구 응답 : {r.status_code}")

    if r.status_code == 200:
        t0 = time.time()
        print("복구 상태 :", wait_green(RECOVER))
        t_green = time.time() - t0
        # 4) 복구 후 검증 : 포인트 수, 검색 결과, 별칭
        after = qdrant_search(RECOVER, qvecs[0], k=5, ef=128)
        n_after = qc.get_collection(RECOVER).points_count
        print(f"복구 후 : {n_after}개, 질의 0 결과 {after}")
        print(f"  포인트 수 일치 : {n_before == n_after}")
        print(f"  검색 결과 일치 : {before == after}")

        # 별칭은 컬렉션 삭제 시 함께 지워졌으며 스냅샷이 자동 복원하지 않으므로 직접 다시 연결합니다
        alias_now = [a.alias_name for a in qc.get_aliases().aliases
                     if a.collection_name == RECOVER]
        print(f"  별칭 복구됨 : {alias_now}")
        for a in alias_before:
            if a not in alias_now:
                ### START CODE HERE ### (RECOVER 컬렉션에 별칭 a 를 다시 연결하세요)
                qc.update_collection_aliases(change_aliases_operations=[
                    CreateAliasOperation(create_alias=CreateAlias(
                        collection_name=RECOVER, alias_name=a))])
                ### END CODE HERE ###
                print(f"  별칭 {a} 를 다시 연결했습니다")

        mttr = time.time() - t_down
        print(f"\n복구 시간 : {mttr:.1f}초 (여러 번 훈련한 평균이 MTTR)")
        print(f"  업로드와 복원 {t_upload:.1f}초, 인덱스 준비 {t_green:.1f}초 "
              f"(장애 전에 만든 스냅샷 생성 {t_snap:.1f}초는 제외)")
    else:
        print("복구 실패. 응답 본문 :", r.text[:300])

    # 5) 서버 재시작 후 영속성 확인
    proc.terminate()
    proc.wait(timeout=30)
    print("\n서버 종료")

    proc, status = start_qdrant()
    print(f"서버 재기동 : {status}")
    qc = QdrantClient(url=QDRANT_URL)
    print("컬렉션 :", sorted(c.name for c in qc.get_collections().collections))
    print("별칭 :", qc.get_aliases())

복구 대상 : docs_v2
복구 전 : 10000개, 질의 0 결과 [5114, 8964, 2664, 144, 6444], 별칭 ['docs_live']
스냅샷 : qdrant_snapshots/docs_v2/docs_v2-2307530473306109-2026-10-22-07-42-47.snapshot (344.3 MB), 파일 있음 True
컬렉션 삭제됨. 존재 여부 False, 별칭 []
복구 응답 : 200
복구 상태 : green
복구 후 : 10000개, 질의 0 결과 [5114, 2664, 8964, 144, 6444]
  포인트 수 일치 : True
  검색 결과 일치 : False
  별칭 복구됨 : []
  별칭 docs_live 를 다시 연결했습니다

복구 시간 : 14.1초 (여러 번 훈련한 평균이 MTTR)
  업로드와 복원 14.0초, 인덱스 준비 0.1초 (장애 전에 만든 스냅샷 생성 6.7초는 제외)

서버 종료
서버 재기동 : 기동 완료
컬렉션 : ['docs_base', 'docs_hybrid', 'docs_m16', 'docs_m2', 'docs_m32', 'docs_m4', 'docs_m8', 'docs_sq', 'docs_v2']
별칭 : aliases=[AliasDescription(alias_name='docs_live', collection_name='docs_v2')]


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 1) 스냅샷 생성
snap = qc.create_snapshot(collection_name=RECOVER)

# 4) 유실된 별칭 재연결
qc.update_collection_aliases(change_aliases_operations=[
    CreateAliasOperation(create_alias=CreateAlias(
        collection_name=RECOVER, alias_name=a))])
```

검증 항목은 포인트 수, 검색 결과, 별칭, 재시작 후 영속성입니다. 하나만 확인하면 불충분합니다.

**스냅샷은 컬렉션의 데이터를 되살리지만 별칭은 되살리지 않습니다.** 컬렉션을 지우는 순간 그 컬렉션을 가리키던 별칭도 함께 사라지고 복구해도 돌아오지 않습니다. 서비스 코드는 별칭만 참조하므로 **데이터를 완벽히 복구하고도 서비스는 여전히 멈춰 있습니다.** 복구 절차서에 별칭을 다시 붙이는 단계가 빠지면 이 사고가 납니다.

**복구 시간**을 여러 번의 훈련에서 측정해 평균을 낸 값이 교재 표 4-1의 MTTR(평균 복구 시간)입니다. 여기서 측정한 것은 컬렉션을 지운 순간부터 별칭을 다시 붙이기까지의 기술적 복구 시간이고 실제 MTTR에는 장애를 알아채고 대응을 시작하기까지의 시간이 더해집니다. 스냅샷이 커지면 업로드와 인덱스 준비 시간이 늘어나므로 이 값을 마무리 과제 12번에 적고 데이터가 늘 때마다 다시 측정합니다.

영속성도 다릅니다. 임베디드 모드에서는 프로세스가 끝나면 접근이 끊겼지만 서버 모드에서는 서버가 재시작해도 데이터가 남아 있습니다.

**[현업 환경 적용]** : 사내에서는 이 절차를 문서로 남기고 정기적으로 훈련해야 합니다. 장애가 났을 때 처음 복구를 시도하면 늦습니다. 마무리 과제 5단계가 그 문서를 쓰는 일입니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

## Step 9. 선택 심화 트랙

트랙 1, 2, 3, 4, 6 가운데 **두 개를 선택**해 수행하고 결과를 공유합니다. (트랙 5 장애 복구 시나리오는 본 실습 Step 8-1로 이동했습니다.)

<div style="font-size: 18px; line-height: 1.6;">

| 트랙 | 주제 | 성격 | 난이도 |
|---|---|---|---|
| 1 | 규모 확장과 인덱스 손익분기점 찾기 | 측정 | 중 |
| 2 | 이진 양자화와 재점수화 | 최적화 | 상 |
| 3 | 멀티테넌시 설계 비교 | 설계 | 중 |
| 4 | 파이프라인 연동과 게이트 기반 전환 | 확장 | 상 |
| 5 | 문서 하나에 벡터가 여럿일 때의 저장 설계(Multi-vector) | 설계 | 상 |

</div>

#### 트랙 1. 데이터 규모별 완전 탐색과 ANN 검색의 지연 시간 교차점

이론 1.2절의 어림셈을 실제 측정값으로 확인해 **완전 탐색보다 ANN 검색이 유리해지는 지점**을 직접 찾습니다.

In [ ]:
ensure_qdrant()

# 실습 : 데이터 규모(5천, 2만, 5만)를 늘려가며 완전 탐색(행렬곱)과 ANN 검색의 지연을 비교하세요

def bench_scale(n):
    sub_chunks = make_chunks(n)
    sub_vecs = np.array(emb.embed_documents(
        [f"passage: {c['text']}" for c in sub_chunks]), dtype=np.float32)

    # 1) 완전 탐색 (Brute-Force)
    t0 = time.time()
    for i in range(len(queries)):
        ### START CODE HERE ### (전체 벡터 행렬 sub_vecs 와 질의 벡터 qvecs[i] 의 내적(@)을 계산하세요)
        scores = sub_vecs @ qvecs[i]
        ### END CODE HERE ###
        if scores is None:
            return None, None
        np.argsort(scores)[::-1][:10]
    bf = (time.time() - t0) / len(queries) * 1000

    # 2) ANN 검색 (HNSW)
    cname = f"scale_{n}"
    reset_collection(cname)
    for s in range(0, n, B):
        e = min(s + B, n)
        qc.upsert(collection_name=cname, points=[
            PointStruct(id=j, vector=sub_vecs[j].tolist(), payload={})
            for j in range(s, e)], wait=(e >= n))
    wait_green(cname)

    t0 = time.time()
    for i in range(len(queries)):
        ### START CODE HERE ### (cname 컬렉션에 hnsw_ef=128 로 상위 10개를 ANN 검색하세요)
        qc.query_points(collection_name=cname, query=qvecs[i].tolist(), limit=10,
                        search_params=SearchParams(hnsw_ef=128))
        ### END CODE HERE ###
    ann = (time.time() - t0) / len(queries) * 1000

    qc.delete_collection(cname)
    return bf, ann


print(f"{'규모':>8} {'완전탐색(ms)':>14} {'ANN검색(ms)':>14} {'비율':>8}")
print("-" * 48)
for n in [2500, 10000, 25000]:
    bf, ann = bench_scale(n)
    if bf is None:
        print("[안내] bench_scale 안의 scores 빈칸을 채운 뒤 실행하세요.")
        break
    print(f"{n:>8} {bf:>14.2f} {ann:>14.2f} {bf/ann:>8.2f}x")

      규모       완전탐색(ms)      ANN검색(ms)       비율
------------------------------------------------
    2500           0.24           2.73     0.09x
   10000           0.97           3.43     0.28x
   25000           3.02           2.82     1.07x

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 1) 완전 탐색 내적 계산
scores = sub_vecs @ qvecs[i]

# 2) ANN 검색 호출
qc.query_points(collection_name=cname, query=qvecs[i].tolist(), limit=10,
                search_params=SearchParams(hnsw_ef=128))
```

ANN 검색에는 네트워크 왕복과 그래프 탐색 오버헤드가 있어 **작은 규모에서는 완전 탐색이 더 빠릅니다.** 규모가 커지면 완전 탐색 시간은 선형으로 늘고 ANN 검색은 거의 늘지 않아 어느 지점에서 교차합니다.

**이 교차점은 이 환경에서 한 번 검색의 지연이 역전되는 지점일 뿐 벡터DB 도입 기준이 아닙니다.** 교차점을 넘어도 완전 탐색이 수십 밀리초 안이면 사용자는 차이를 느끼지 못하고 이론 1.1·1.3절의 말대로 동시 사용자가 적으면 100만 개 안팎까지 넘파이 배열과 곱셈으로 충분합니다. 도입을 가르는 것은 이론 1.2\~1.3절의 어림셈과 동시성이고 이론 4.7절의 2번 질문이 이것입니다.

그래도 벡터DB를 쓰는 이유는 속도 말고도 영속성, 필터, 증분 갱신, 여러 서비스 공유, 모니터링에 있습니다. 프라이빗 RAG 노트에서 12개 청크에 Qdrant를 쓴 것도 이 기능들을 배우기 위해서였습니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 2. 이진 양자화(BQ)와 오버샘플링·재점수화 효과

이론 3.3절의 이진 양자화는 32분의 1로 줄이는 대신 많이 놓칩니다. 이론 3.4절의 오버샘플링과 재점수화로 보완합니다.

In [51]:
ensure_qdrant()

# 실습 : 이진 양자화(BinaryQuantization)를 적용하고, 재점수화(rescore)와 오버샘플링(oversampling) 효과를 측정하세요

from qdrant_client.models import (BinaryQuantization, BinaryQuantizationConfig,
                                  QuantizationSearchParams)

bq = BinaryQuantization(binary=BinaryQuantizationConfig(always_ram=True))
create_and_index("docs_bq", quantization=bq)


def search_bq(qv, k=30, ef=128, rescore=False, oversampling=1.0):
    ### START CODE HERE ### (rescore 와 oversampling 을 지정한 QuantizationSearchParams 생성)
    q_params = QuantizationSearchParams(rescore=rescore, oversampling=oversampling)
    ### END CODE HERE ###
    if q_params is None:
        return []
    hits = qc.query_points(
        collection_name="docs_bq", query=qv.tolist(), limit=k,
        search_params=SearchParams(hnsw_ef=ef, quantization=q_params),
    ).points
    return [h.id for h in hits]


if not search_bq(qvecs[0]):
    print("[안내] search_bq 안의 q_params 빈칸을 채운 뒤 실행하세요.")
else:
    print(f"{'조건':<30} {'놓치는 비율':>12} {'ms/질의':>10}")
    print("-" * 54)
    for label, kw in [
        ("이진 양자화, 재점수화 없음", dict(rescore=False)),
        ("이진 양자화 + 오버샘플링 2배·재점수화", dict(rescore=True, oversampling=2.0)),
        ("이진 양자화 + 오버샘플링 4배·재점수화", dict(rescore=True, oversampling=4.0)),
    ]:
        t0 = time.time()
        misses = [miss_rate(search_bq(qvecs[i], **kw), ground_truth[i])
                  for i in range(len(queries))]
        ms = (time.time() - t0) / len(queries) * 1000
        print(f"{label:<30} {np.mean(misses):>12.3f} {ms:>10.2f}")

    print("\n크기 비교")
    for n in ["docs_base", "docs_sq", "docs_bq"]:
        if qc.collection_exists(n):
            print(f"  {n:<12} {collection_size_mb(n):>8.1f} MB")

조건                                   놓치는 비율      ms/질의
------------------------------------------------------
이진 양자화, 재점수화 없음                       0.956       2.61
이진 양자화 + 오버샘플링 2배·재점수화                0.627       2.63
이진 양자화 + 오버샘플링 4배·재점수화                0.448       2.73

크기 비교
  docs_base       437.1 MB
  docs_sq         344.3 MB
  docs_bq         478.5 MB


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
q_params = QuantizationSearchParams(
    rescore=rescore, oversampling=oversampling
)
```

벡터를 1비트로 줄이면 정보가 많이 사라지므로 **재점수화 없는 이진 양자화는 훨씬 많이 놓칩니다.**

`oversampling`이 오버샘플링 배수입니다. 값이 2면 필요한 개수의 두 배를 압축 벡터로 뽑고 원본 벡터로 점수를 다시 계산해 상위를 고릅니다. 값을 올리면 덜 놓치지만 지연이 늘어납니다. 이론 3.4절의 곡선입니다. Step 4.1과 같은 방법으로 꺾이는 지점을 찾습니다.

크기 비교도 Step 6과 같게 읽습니다. 원본을 함께 들고 있어야 재점수화를 할 수 있으므로 **디스크는 줄지 않습니다.** 32분의 1이 되는 것은 메모리에 올려두는 압축본입니다.

**저비용 연산으로 넓게 추리고 고비용 연산으로 정밀화하기**는 이 과정에서 반복해 만난 구조입니다. 하이브리드 검색의 결과 합치기, 리랭커를 쓰는 2단계 검색, 중복 제거의 두 단계가 모두 같은 형태였습니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 3. 멀티테넌시 설계(단일 컬렉션 필터 vs 부서별 컬렉션) 비교

지난 노트의 권한 기반 검색과 이어지는 부서별 데이터 분리 방법을 비교합니다.

In [ ]:
ensure_qdrant()

# 실습 : 멀티테넌시 두 방식(A: 단일 컬렉션 + 필터 vs B: 부서별 물리 분리 컬렉션)을 구현하고 비교하세요

TARGET_DEPT = "보안팀"
dept_chunks = [c for c in chunks if c["dept"] == TARGET_DEPT]
print(f"{TARGET_DEPT} 청크 {len(dept_chunks)}개")

### START CODE HERE ### (방식 A: dept 필드가 TARGET_DEPT 와 일치하는 Filter 작성)
flt_a = Filter(must=[FieldCondition(key="dept", match=MatchValue(value=TARGET_DEPT))])
### END CODE HERE ###

if flt_a is None:
    print("[안내] flt_a 빈칸을 채운 뒤 실행하세요.")
else:
    t0 = time.time()
    for i in range(len(queries)):
        qdrant_search("docs_base", qvecs[i], k=10, ef=128, flt=flt_a)
    ms_a = (time.time() - t0) / len(queries) * 1000

    # B : 부서별 전용 컬렉션 분리
    cname_b = "dept_security"
    reset_collection(cname_b)
    qc.upsert(collection_name=cname_b, points=[
        PointStruct(id=c["id"], vector=vectors[c["id"]].tolist(),
                    payload={"text": c["text"], "doc_type": c["doc_type"]})
        for c in dept_chunks], wait=True)
    wait_green(cname_b)

    t0 = time.time()
    for i in range(len(queries)):
        ### START CODE HERE ### (방식 B: 필터 없이 부서 전용 컬렉션 cname_b 에서 검색)
        qc.query_points(collection_name=cname_b, query=qvecs[i].tolist(), limit=10,
                        search_params=SearchParams(hnsw_ef=128))
        ### END CODE HERE ###
    ms_b = (time.time() - t0) / len(queries) * 1000

    print(f"\n{'방식':<24} {'ms/질의':>10} {'격리':>8} {'운영복잡도':>12}")
    print("-" * 58)
    print(f"{'A. 단일+필터':<24} {ms_a:>10.2f} {'논리적':>8} {'낮음':>12}")
    print(f"{'B. 부서별 컬렉션':<24} {ms_b:>10.2f} {'물리적':>8} {'높음':>12}")

보안팀 청크 1428개

방식                            ms/질의       격리        운영복잡도
----------------------------------------------------------
A. 단일+필터                       3.03      논리적           낮음
B. 부서별 컬렉션                     2.64      물리적           높음

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 방식 A: 부서 필터 정의
flt_a = Filter(must=[FieldCondition(key="dept", match=MatchValue(value=TARGET_DEPT))])

# 방식 B: 물리 분리된 부서 전용 컬렉션 검색
qc.query_points(collection_name=cname_b, query=qvecs[i].tolist(), limit=10,
                search_params=SearchParams(hnsw_ef=128))
```

**세 방식의 판단 기준**

**A (단일 + 필터)** : 운영이 가장 단순하고 부서 간 통합 검색이 됩니다. 대신 격리가 논리적일 뿐이라 **코드에서 필터를 빼먹으면 즉시 유출**됩니다.

**B (컬렉션 분리)** : 물리적으로 격리되어 코드 실수로 유출되지 않습니다. 대신 부서가 늘 때마다 컬렉션이 늘고 통합 검색을 하려면 여러 컬렉션을 조회해 합쳐야 합니다. 작은 그래프가 여럿 생겨 메모리 효율도 떨어집니다.

**C (별칭)** : 컬렉션은 하나이고 부서별 별칭을 만들어 접근을 나눕니다. 다만 Qdrant의 별칭은 필터를 포함하지 않으므로 이 방식만으로는 격리가 되지 않습니다. **별칭은 재인덱싱 전환용이지 권한 분리용이 아닙니다.**

**따라서 실무에서는 데이터 민감도에 따라 두 방식을 혼합합니다.** 공개·사내 일반 문서는 단일 컬렉션(A)에 두고 필터로 구분하되 유출 시 피해가 큰 대외비·기밀 문서는 별도 컬렉션(B)으로 물리 분리해 운영 복잡도를 감수합니다(이론 4.7절 Quiz).

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 4. 사전 검증 게이트 통과 시에만 수행하는 안전한 별칭 전환

지난 노트의 파이프라인과 이번 노트의 별칭 전환을 결합합니다. 검사 항목은 이론 6.6절의 네 가지 그대로입니다.

In [ ]:
ensure_qdrant()

# 실습 : 새 컬렉션을 인덱싱하고 4가지 배포 게이트(deploy_gate)를 통과했을 때만 별칭을 전환하세요

FILTER_FIELDS = [("dept", PayloadSchemaType.KEYWORD),
                 ("doc_type", PayloadSchemaType.KEYWORD),
                 ("year", PayloadSchemaType.INTEGER)]


def index_to_new_collection(name, items, vecs, with_payload_index=True):
    reset_collection(name)
    if with_payload_index:
        for field, schema in FILTER_FIELDS:
            qc.create_payload_index(collection_name=name, field_name=field,
                                    field_schema=schema)
    for s in range(0, len(items), B):
        e = min(s + B, len(items))
        qc.upsert(collection_name=name, points=[
            PointStruct(id=c["id"], vector=vecs[c["id"]].tolist(),
                        payload={k: c[k] for k in ("text", "dept", "doc_type", "topic", "year")})
            for c in items[s:e]], wait=(e >= len(items)))
    wait_green(name)
    return qc.get_collection(name).points_count


def avg_miss(target):
    return float(np.mean([miss_rate(qdrant_search(target, qvecs[i], ef=128),
                                    ground_truth[i]) for i in range(len(queries))]))


def deploy_gate(new_name, expected_min, expected_max, baseline=None, margin=0.10):
    """네 가지를 검사합니다. 판정은 불합격이 기본값입니다"""
    checks = []
    info = qc.get_collection(new_name)

    ### START CODE HERE ### (게이트 판정 조건식 3곳을 채우세요)
    # 1) 포인트 수 n 이 expected_min 이상이고 expected_max 이하인지 검사 (상·하한 동시 검사)
    n = info.points_count
    ok_count = (expected_min <= n <= expected_max)
    checks.append(("포인트 수", bool(ok_count),
                   f"{n} (기대 {expected_min} 이상 {expected_max} 이하)"))

    # 2) 필수 payload 필드 (미리 작성됨)
    sample = qc.scroll(collection_name=new_name, limit=20, with_payload=True)[0]
    required = {"text", "dept", "doc_type", "year"}
    ok_fields = bool(sample) and all(required <= set(p.payload.keys()) for p in sample)
    checks.append(("payload 필드", ok_fields, f"필수 {sorted(required)}"))

    # 3) 대표 질문에서 기존 인덱스(base_mr) + 마진(margin) 이하로 놓치는지 검사
    mr = avg_miss(new_name)
    base_mr = avg_miss(baseline) if baseline and qc.collection_exists(baseline) else None
    if base_mr is None:
        checks.append(("놓치는 비율", mr <= 0.10, f"{mr:.3f} (비교 대상 없음, 기준 0.10)"))
    else:
        ok_quality = (mr <= base_mr + margin)
        checks.append(("놓치는 비율", bool(ok_quality),
                       f"{mr:.3f} (기존 {base_mr:.3f} + 마진 {margin})"))

    # 4) 필터 필드 집합(want)이 실제 생성된 payload 인덱스 집합(indexed)의 부분집합(<=)인지 검사
    indexed = set(info.payload_schema)
    want = {f for f, _ in FILTER_FIELDS}
    ok_index = (want <= indexed)
    checks.append(("필터 인덱스", bool(ok_index), f"{sorted(indexed) or '없음'}"))
    ### END CODE HERE ###

    print(f"{'검사':<14} {'결과':>6}  상세")
    print("-" * 56)
    for name, ok, detail in checks:
        print(f"{name:<14} {'통과' if ok else '실패':>6}  {detail}")
    return all(ok for _, ok, _ in checks)


print("[일부러 필터 인덱스를 빼고 만든 컬렉션]")
index_to_new_collection("docs_nogate", chunks, vectors, with_payload_index=False)
print("게이트 결과 :",
      deploy_gate("docs_nogate", int(N * 0.9), int(N * 1.1), baseline="docs_base"), "\n")

print("[정상 절차로 만든 컬렉션]")
n = index_to_new_collection("docs_v3", chunks, vectors)
print(f"새 컬렉션 인덱싱 완료 : {n}개\n")

if deploy_gate("docs_v3", int(N * 0.9), int(N * 1.1), baseline="docs_base"):
    qc.update_collection_aliases(change_aliases_operations=[
        DeleteAliasOperation(delete_alias=DeleteAlias(alias_name=ALIAS)),
        CreateAliasOperation(create_alias=CreateAlias(
            collection_name="docs_v3", alias_name=ALIAS)),
    ])
    print("\n게이트 통과. 별칭을 전환했습니다 :", qc.get_aliases())
else:
    print("\n게이트 실패(또는 빈칸 미완성). 기존 인덱스를 유지합니다")

qc.delete_collection("docs_nogate")

[일부러 필터 인덱스를 빼고 만든 컬렉션]
검사                 결과  상세
--------------------------------------------------------
포인트 수              통과  10000 (기대 9000 이상 11000 이하)
payload 필드         통과  필수 ['dept', 'doc_type', 'text', 'year']
놓치는 비율             통과  0.017 (기존 0.012 + 마진 0.1)
필터 인덱스             실패  없음
게이트 결과 : False 

[정상 절차로 만든 컬렉션]
새 컬렉션 인덱싱 완료 : 10000개

검사                 결과  상세
--------------------------------------------------------
포인트 수              통과  10000 (기대 9000 이상 11000 이하)
payload 필드         통과  필수 ['dept', 'doc_type', 'text', 'year']
놓치는 비율             통과  0.012 (기존 0.012 + 마진 0.1)
필터 인덱스             통과  ['dept', 'doc_type', 'year']

게이트 통과. 별칭을 전환했습니다 : aliases=[AliasDescription(alias_name='docs_live', collection_name='docs_v3')]

True

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
ok_count = (expected_min <= n <= expected_max)
# ...
ok_quality = (mr <= base_mr + margin)
# ...
ok_index = (want <= indexed)
```

**인덱싱과 전환을 분리하고 그 사이에 게이트를 두는 것**이 실무 배포의 표준 형태입니다.

지난 노트의 파이프라인 게이트와 같은 발상입니다. 다만 여기서는 게이트가 실패해도 파이프라인이 멈추지 않고 기존 인덱스가 유지되어 사용자는 아무것도 못 느낍니다(이론 6.6절).

**코드에서 볼 것**

**포인트 수 검사에 상한을 둔 것.** 하한만 검사하면 지난 노트 Quiz의 "청크가 세 배가 된" 사고를 통과시킵니다.

**품질 기준을 고정값으로 두지 않고 지금 서비스 중인 인덱스와 비교한 것.** 이론 6.6절은 "대표 질문들의 결과가 **기존만큼** 나오는가"라고 씁니다. 절대값으로 고정하면 데이터가 조금 달라졌을 때 멀쩡한 인덱스를 막거나 나빠진 인덱스를 통과시킵니다. 별칭이 가리키는 곳을 그대로 비교 대상으로 넘기면 그 기준이 언제나 최신입니다.

그래프를 만드는 순서가 매번 다르고 우리 합성 문서는 서로 비슷해 점수가 촘촘하므로 **같은 데이터를 다시 인덱싱해도 놓치는 비율이 조금씩 달라집니다.** 마진이 이 흔들림보다 작으면 게이트가 멀쩡한 인덱스를 막습니다.

그래서 마진을 정하려면 **흔들림의 크기를 먼저 측정해야 합니다.** 같은 설정으로 두 번 인덱싱해 두 값의 차이를 봅니다.

흔들림을 줄이려면 마진을 키우기보다 **대표 질문을 늘립니다.** 질문이 여덟 개뿐이면 문서 하나 차이가 0.0125라 게이트가 매번 다른 답을 냅니다. 그래서 Step 1에서 질문 수를 늘려 잡았습니다. 이론 5.5절이 대표 질문을 등록해두라고 한 이유도 같습니다.

**일부러 실패하는 컬렉션을 먼저 검사한 것.** 폴백이든 게이트든 1단계가 실패를 실제로 감지하는지 확인하지 않으면 그 장치가 동작하는지 알 수 없습니다. 필터 인덱스 검사가 없었다면 `docs_nogate`도 그대로 통과했을 것입니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 5. 장애 복구 시나리오
*(안내: 이 트랙은 본 실습 **Step 8-1**로 이동하여 필수 과정으로 진행했습니다.)*

#### 트랙 6. 명명된 다중 벡터(요약·조각) 저장과 2단계 검색

이론 6.7절의 저장 설계입니다. 멀티모달 RAG 노트에서 본 **late interaction**을 쓰려면 문서 하나에 조각 벡터를 여럿 들고 있어야 합니다. 그 저장이 벡터DB 안에서 어떤 형태가 되는지 확인합니다. 과제는 이론 6.7절에서 인덱싱 전에 확인하라고 한 네 가지에 코드로 답하는 일입니다.

In [52]:
ensure_qdrant()

# 실습 : 조각 벡터를 묶어 저장(MultiVector)하고 2단계 검색(Prefetch -> Rerank)을 완성하세요

from qdrant_client.models import (MultiVectorConfig, MultiVectorComparator, Prefetch)

SUB = 2500  # 실습 경량화 (2,500개 문서)      # 조각 벡터는 개수가 몇 배가 되므로 앞 8,000개 문서만 씁니다
sub = chunks[:SUB]


def split_parts(text, limit=6):
    """문장 단위로 조각을 냅니다."""
    parts = [s.strip() for s in text.split(". ") if s.strip()]
    return parts[:limit]


part_texts, owner = [], []
for c in sub:
    for s in split_parts(c["text"]):
        part_texts.append(f"passage: {s}")
        owner.append(c["id"])

part_vecs = np.array(emb.embed_documents(part_texts), dtype=np.float32)
print(f"문서 {len(sub)}개 -> 조각 {len(part_vecs)}개 "
      f"(문서당 {len(part_vecs)/len(sub):.1f}개)")

by_doc = {}
for v, oid in zip(part_vecs, owner):
    by_doc.setdefault(oid, []).append(v.tolist())

MULTI = "docs_multi"
if qc.collection_exists(MULTI):
    qc.delete_collection(MULTI)

### START CODE HERE ### (1. 멀티벡터 MAX_SIM 설정과 2. 1차 요약 벡터 Prefetch 설정 2곳을 채우세요)
# 1) 조각 벡터끼리 비교할 때 최대 유사도 합산 방식(MultiVectorComparator.MAX_SIM)을 지정합니다
mv_comparator = MultiVectorComparator.MAX_SIM

# 2) 2단계 검색에서 요약 벡터("summary")로 상위 100개 후보를 먼저 뽑는 Prefetch 함수를 정의합니다
def make_summary_prefetch(qv):
    return Prefetch(query=qv.tolist(), using="summary", limit=100)
### END CODE HERE ###

if mv_comparator is None or make_summary_prefetch(qvecs[0]) is None:
    print("[안내] mv_comparator 와 make_summary_prefetch 빈칸을 채운 뒤 다시 실행하세요.")
else:
    qc.create_collection(
        collection_name=MULTI,
        vectors_config={
            "summary": VectorParams(size=DIM, distance=Distance.COSINE),
            "parts": VectorParams(
                size=DIM, distance=Distance.COSINE,
                multivector_config=MultiVectorConfig(comparator=mv_comparator),
                # 조각에는 그래프를 만들지 않습니다(m=0). 후보의 순위를 다시 매기는 데만 씁니다
                hnsw_config=HnswConfigDiff(m=0)),
        })

    MB = 250
    for s in range(0, len(sub), MB):
        qc.upsert(collection_name=MULTI, points=[
            PointStruct(id=c["id"],
                        vector={"summary": vectors[c["id"]].tolist(),
                                "parts": by_doc[c["id"]]},
                        payload={"dept": c["dept"]})
            for c in sub[s:s + MB]], wait=(s + MB >= len(sub)))
    print("인덱스 상태 :", wait_green(MULTI), "포인트",
          qc.get_collection(MULTI).points_count)

    create_and_index("docs_single", items=sub, vecs=vectors)

    def timed(fn, repeat=3):
        t0 = time.time()
        for _ in range(repeat):
            for i in range(len(queries)):
                fn(qvecs[i])
        return (time.time() - t0) / (repeat * len(queries)) * 1000

    one = timed(lambda qv: qc.query_points(
        collection_name=MULTI, query=qv.tolist(), using="summary", limit=10))
    two = timed(lambda qv: qc.query_points(
        collection_name=MULTI,
        prefetch=make_summary_prefetch(qv),
        query=[qv.tolist()], using="parts", limit=10))
    raw = timed(lambda qv: qc.query_points(
        collection_name=MULTI, query=[qv.tolist()], using="parts", limit=10), repeat=1)

    print(f"\n{'방식':<28} {'ms/질의':>10}")
    print("-" * 40)
    print(f"{'1단계만 (요약 벡터)':<28} {one:>10.2f}")
    print(f"{'2단계 (요약 -> 조각)':<28} {two:>10.2f}")
    print(f"{'조각만 (후보 없이 전부)':<28} {raw:>10.2f}")

    print(f"\n{'컬렉션':<14} {'크기(MB)':>10}")
    print("-" * 26)
    for n in ["docs_single", MULTI]:
        print(f"{n:<14} {collection_size_mb(n):>10.1f}")

문서 2500개 -> 조각 7500개 (문서당 3.0개)
인덱스 상태 : green 포인트 2500

방식                                ms/질의
----------------------------------------
1단계만 (요약 벡터)                       2.29
2단계 (요약 -> 조각)                     3.41
조각만 (후보 없이 전부)                     2.22

컬렉션                크기(MB)
--------------------------
docs_single         344.1
docs_multi          516.1


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
mv_comparator = MultiVectorComparator.MAX_SIM

def make_summary_prefetch(qv):
    return Prefetch(query=qv.tolist(), using="summary", limit=100)
```

**이론 6.7절의 확인 사항에 대한 답**

**1. 벡터 여럿을 문서 하나로 묶을 수 있는가.** 포인트 하나에 이름 붙은 벡터를 여럿 두고 그중 하나를 멀티벡터로 선언했습니다. 묶지 못하는 제품이면 조각마다 포인트를 만들고 결과를 문서 단위로 합치는 코드를 우리가 짜야 합니다.

**2. 후보를 처음부터 뽑아주는가, 순위만 다시 매기는가.** 조각 벡터에 `m=0`을 준 것이 그 답입니다. 그래프를 안 만들었으므로 조각 벡터로는 후보를 뽑을 수 없고 **요약 벡터가 뽑아준 후보의 순위만 다시 매깁니다.** "조각만" 줄의 지연을 봅니다. 후보 없이 조각으로 바로 찾으면 완전 탐색이 되어 크게 느립니다.

**3. 압축을 걸 수 있는가.** 조각 벡터에도 Step 6과 트랙 2의 양자화를 그대로 겁니다. 개수가 몇 배이므로 줄이는 절대량이 가장 큰 부분입니다.

**4. 어림셈에 조각 수를 곱해도 서버에 들어가는가.** 크기 비교가 그 답입니다. 문서 수는 같은데 저장이 몇 배로 늘었습니다. 이론 1.2절의 어림셈에 **문서당 조각 수를 그대로 곱하면** 됩니다.

넣는 코드에서 한 번에 보내는 양을 줄인 것도 같은 이야기입니다. 문서 하나가 벡터 여럿이 되면 **요청 본문부터 몇 배가 됩니다.** 실제 late interaction 모델은 조각이 수십 개이므로 인덱싱 코드를 그대로 가져다 쓰면 요청 크기 상한에 먼저 걸립니다.

2단계 구조는 이론 3.4절과 똑같이 **저비용 연산으로 넓게 추리고 고비용 정밀 연산으로 좁힙니다.** 오늘만 세 번째입니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

## Step 10. 서버 프로세스 종료 및 리소스 정리

In [ ]:
# 최종 상태 확인
print(f"{'컬렉션':<16} {'포인트':>8} {'크기(MB)':>10}  필터 인덱스")
print("-" * 58)
for c in sorted(qc.get_collections().collections, key=lambda x: x.name):
    info = qc.get_collection(c.name)
    print(f"  {c.name:<14} {info.points_count:>8} {collection_size_mb(c.name):>10.1f}  "
          f"{sorted(info.payload_schema)}")
print(f"\n별칭 : {qc.get_aliases()}")

컬렉션                   포인트     크기(MB)  필터 인덱스
----------------------------------------------------------
  dept_security      1428        6.1  []
  docs_base         10000       43.6  ['dept', 'doc_type', 'year']
  docs_bq           10000       42.8  []
  docs_hybrid         500        2.5  []
  docs_m16          10000       42.3  []
  docs_m32          10000       42.3  []
  docs_m8           10000       42.3  []
  docs_multi         2500       35.2  []
  docs_single        2500       10.5  []
  docs_sq           10000       42.3  []
  docs_v2           10000       42.3  []
  docs_v3           10000       43.6  ['dept', 'doc_type', 'year']

별칭 : aliases=[AliasDescription(alias_name='docs_live', collection_name='docs_v3')]

In [ ]:
# 실습이 끝나면 서버를 정리합니다
proc.terminate()
try:
    proc.wait(timeout=30)
except subprocess.TimeoutExpired:
    proc.kill()
print("Qdrant 서버 종료")

Qdrant 서버 종료

## Step 11. 실습 결과 점검과 정리

### Step 10. 결과 점검과 정리

(선택 대상: 트랙 1 데이터 규모별 완전 탐색과 ANN 검색의 지연 시간 교차점 / 트랙 2 이진 양자화 BQ와 오버샘플링·재점수화 효과 / 트랙 3 멀티테넌시 설계: 단일 컬렉션 필터 vs 부서별 컬렉션 분리 / 트랙 4 사전 검증 게이트 통과 시에만 수행하는 안전한 별칭 전환 / 트랙 6 명명된 다중 벡터 저장과 2단계 검색)
각자 수행한 두 트랙을 공유합니다. 아래 형식으로 정리해주세요.
1. **선택한 트랙과 이유**
2. **측정 결과** : 설정 변화(`m`, `ef_construct`, `hnsw_ef`, 양자화 `SQ8`/`BQ`, 오버샘플링 배수, 하이브리드 `RRF`)에 따른 지연 시간(`p50`/`p95`), 완전 탐색 대비 놓친 비율(`Miss Rate`), 메모리 사용량의 변화
3. **인덱스와 양자화의 한계를 만난 지점** : 낮은 통과율 필터에서의 지연 급증, 고유 코드·조항 번호에서 밀집 벡터 단독 검색의 한계, 이진 양자화(`BQ`) 단독 사용 시 정밀도 급락, `m` 증가의 수확 체감 등 구체적으로
4. **이 벡터DB를 사내에 배포한다면** : 사내 문서 규모와 동시 접속자 수(`QPS`), 서버 RAM 예산(`estimate_capacity`)에 맞춘 인덱스·양자화·멀티테넌시 설정과 무중단 재인덱싱(`deploy_gate` + 별칭 전환) 주기

공유가 끝나면 오늘 만든 것을 한 문장으로 정리해봅시다.
우리는 서버 메모리 예산을 역산해 완전 탐색 기준선 위에서 그래프 탐색 폭과 양자화로 속도·메모리·정확도를 맞바꾸는 지점을 실측하고, 필터 인덱스와 하이브리드 검색을 결합한 뒤, 검증 게이트와 별칭 전환·스냅샷 복구로 검색을 멈추지 않고 인덱스를 교체하는 절차를 만들었습니다.
여기서 질문이 하나 생깁니다. 이 모든 것을 통틀어 부르는 이름이 있을까요? 단순히 문서를 임베딩해서 넣는 저장소 사용법도 아니고 일회성 벤치마크도 아니고, 검색 품질과 자원 비용·무중단 가용성을 숫자로 통제하는 이 운영 체계 전체를 말입니다.

<div style="font-size: 18px; line-height: 1.6;">

| 핵심 설계 축 | 수행 내용 | 핵심 검증 지표 / 기준 |
| :--- | :--- | :--- |
| **용량 역산 및 기준선 (Step 1~3)** | RAM 예산 역산(`estimate_capacity`) 및 넘파이 완전 탐색(Exact k-NN) 구현 | 100% 정답 기준선(`Recall@10 = 1.0`) 및 메모리 상한선 |
| **HNSW & 하이브리드 검색 (Step 4~5.1)** | `m`·`hnsw_ef` 트레이드오프 계측, `Payload` 인덱스, `Dense+Sparse RRF` | `p50`/`p95` 지연 시간 및 미검출률(`Miss Rate`) 변곡점 |
| **벡터 양자화 및 보정 (Step 6, 트랙 2)** | `FP32` 대비 `SQ8`(1/4)·`BQ`(1/32) 압축 및 오버샘플링+`Rescore` | 메모리 75~97% 절감 및 정밀도 복원율 |
| **무중단 운영 및 복구 (Step 7~8)** | 사전 검증 게이트(`deploy_gate`) + 별칭(`Alias`) 원자적 전환 + 스냅샷 복원 | 다운타임 `0초` 전환 및 평균 복구 시간(`MTTR`) |

</div>

## 마무리 과제. 온프레미스 벡터 DB 구축 및 운영 계획서 작성

여러분이 가정한 사내 프로젝트의 벡터DB 운영 계획을 작성합니다.

**1단계. 규모 산정**

1. 문서 수, 문서당 평균 청크 수, 임베딩 차원을 정하고 총 벡터 수를 계산합니다.  
2. 이론 1.2절의 두 줄로 추정합니다. **벡터 수 × 차원 ÷ 100억**이 한 번 검색에 걸리는 초이고 **벡터 수 × 차원 × 4바이트**가 필요한 메모리입니다.  
3. 여기에 이웃 목록의 메모리를 더합니다. 이론 3.1절에서 본 대로 벡터 본체보다 훨씬 작지만 m을 올리면 같이 늡니다.  
4. 1년 후 예상 규모로 다시 계산하고 압축을 켰을 때의 메모리도 함께 계산합니다. 스칼라 양자화는 4분의 1, 이진 양자화는 32분의 1입니다.

**2단계. 튜닝 실험**

5. 여러분 데이터로 ef(hnsw_ef) 곡선을 측정합니다. 최소 5개 값.  
6. 놓치는 비율이 더 안 떨어지는 지점을 찾아 운영값을 결정하고 근거를 적습니다.  
7. m을 2개 이상 비교해 인덱싱 시간과 놓치는 비율을 기록합니다.  
8. **대표 질문을 몇 개로 잡았는지 함께 적습니다.** 질문 수가 측정의 해상도입니다. 그보다 작은 차이는 읽을 수 없습니다.

**3단계. 필터 설계**

9. 필요한 필터 필드를 나열하고 payload 인덱스 대상을 결정합니다.  
10. 가장 좁은 필터 조건에서 인덱스 전후 성능을 측정합니다. 조건 만족 개수를 함께 적습니다.

**4단계. 운영 계획**

11. 운영 계획 표를 채웁니다. 메모리 항목에는 1단계에서 계산한 값을 그대로 옮겨 적고 어떤 식으로 나온 값인지 한 줄로 적습니다.

<div style="font-size: 18px; line-height: 1.6;">

| 항목 | 계획 |
|---|---|
| 배포 형태 | |
| 서버 사양 (메모리, CPU, 디스크) | |
| 압축 적용 여부와 근거 | |
| 백업 주기와 보관 기간 | |
| 재인덱싱 절차 | |
| 모니터링 지표와 임계값 | |
| 멀티테넌시 방식과 근거 | |

</div>

**5단계. 복구 훈련 기록**

12. Step 8-1처럼 스냅샷 생성부터 복구 검증까지 실제로 수행하고 각 단계의 소요 시간과 복구 시간(MTTR)을 기록합니다.  
13. 다른 사람이 따라 할 수 있는 수준으로 복구 절차서를 작성합니다. **별칭을 다시 붙이는 단계를 빠뜨리지 않습니다.**

**평가 기준**

설정값을 정했는지가 아니라 **근거와 검증**으로 평가합니다. ef를 128로 정했다면 왜 그 값인지 곡선으로 보여야 하고 압축을 안 쓰기로 했다면 메모리 계산으로 근거를 대야 합니다.

**복구를 실제로 해봤는지**가 운영 계획의 진짜 기준입니다. 복구해보지 않은 백업은 백업이 아닙니다.